# Analysing the CW attack on the contrastive model — learning rates and losses

The laboratory notebook for the sine-based CW attack against `combined_model`: learning-rate schedules and grids, the old vs new sine function, rounding at the end vs in the loop, and a close look at the loss itself — modifier trajectories over iterations, and a gallery of candidate penalty functions with their gradients (the same gallery the σ attack's loss was eventually chosen from).

**Where this fits:** this section tests a different defence idea — train the detector with **supervised contrastive learning** so its feature space separates the classes more cleanly, then attack the result with the machinery of `03-attacks/`. The CW machinery aimed at these models continues the lineage of `03-attacks/cw_sine_lineage.ipynb`.

*Consolidated from `analysis_LR_L2_with_penalty.ipynb`, `analysis_Loss.ipynb` and `loss_functions.ipynb` (originally in `CW-CL`). Grid repetitions are shown once per configuration. Outputs cleared.*

## Step 1 — Setup and the rounding/evaluation harness

# Load dataset and libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader,WeightedRandomSampler
from sklearn.model_selection import train_test_split
from scipy import sparse
import gdown
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,balanced_accuracy_score
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve, auc
import time
import os
import math
from sklearn.manifold import TSNE
import seaborn as sns
import pandas as pd
import tqdm
import pickle

torch.manual_seed(0)
np.random.seed(0)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
download_links = ['https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y2.npz',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/insertion_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/removal_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/adverserial_attacks_functions.py',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_AT_rFGSM.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/best_model_AT_imax.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/data/model_AT_max%20(4).pth',
]

output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
from adverserial_attacks_functions import *

In [ ]:
# Open the .pkl file
with open('/content/insertion_array.pkl', 'rb') as f:
    # Load the object
    insertion_array = pickle.load(f)

# Close the file
f.close()

insertion_array = torch.tensor(insertion_array, dtype=torch.uint8).to(device)
print(len(insertion_array))

# Open the .pkl file
with open('/content/removal_array.pkl', 'rb') as f:
    # Load the object
    removal_array = pickle.load(f)

# Close the file
f.close()

removal_array = torch.tensor(removal_array, dtype=torch.uint8).to(device)
print(len(removal_array))

In [ ]:
# Load dataset
X_train = sparse.load_npz("/content/sparse_matrix_0.npz").toarray()
X_val = sparse.load_npz("/content/sparse_matrix_1.npz").toarray()
X_test = sparse.load_npz("/content/sparse_matrix_2.npz").toarray()

X_train = torch.tensor(X_train, dtype=torch.int8)
X_val = torch.tensor(X_val, dtype=torch.int8)
X_test = torch.tensor(X_test, dtype=torch.int8)


y_train = sparse.load_npz("/content/sparse_matrix_y0.npz").toarray().reshape((-1, 1))
y_val = sparse.load_npz("/content/sparse_matrix_y1.npz").toarray().reshape((-1, 1))
y_test = sparse.load_npz("/content/sparse_matrix_y2.npz").toarray().reshape((-1, 1))

y_train = torch.tensor(y_train, dtype=torch.int8)
y_val = torch.tensor(y_val, dtype=torch.int8)
y_test = torch.tensor(y_test, dtype=torch.int8)


print("Shapes:")
print("x_train:", X_train.shape)
print("x_val:", X_val.shape)
print("x_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

In [ ]:
# number of benigns and malicious sample in training dataset
n_ben = (y_train.squeeze()== 0).sum().item()
n_mal = (y_train.squeeze()== 1).sum().item()

# Combine features and labels into datasets
# we use dtype=torch.int8, for Memory-Efficient here, later we will convert to float
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

In [ ]:
# Create an instance of your model
model_AT_rFGSM = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_rFGSM.load_state_dict(torch.load('model_AT_rFGSM.pth', map_location=torch.device(device)))

# Create an instance of your model
model_AT_imax = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_imax.load_state_dict(torch.load('best_model_AT_imax.pth', map_location=torch.device(device)))

In [ ]:
# Define the Linear Classifier(NO NEED)
class LinearClassifier(nn.Module):
    def __init__(self, dim_in=200, num_classes=2):
        super(LinearClassifier, self).__init__()
        self.fc = nn.Linear(dim_in, num_classes)

    def forward(self, x):
        return self.fc(x)


# Define the CoreModel
class CoreModel(nn.Module):
    def __init__(self):
        super(CoreModel, self).__init__()
        self.fc1 = nn.Linear(10000, 200)
        self.dropout1 = nn.Dropout(0.6)
        self.fc2 = nn.Linear(200, 200)
        self.dropout2 = nn.Dropout(0.6)
        self.fc3 = nn.Linear(200, 2)

    def forward(self, x, return_z=False):
        x = F.relu(self.fc1(x))
        #x = self.dropout1(x)
        z = F.relu(self.fc2(x))
        z2 = self.dropout2(z)
        output  = self.fc3(z2)
        if return_z:
            return output, z

        return output

    def forward_from_z(self, z):
        output = self.fc3(z)
        return output

# Define the Projection Head
class ProjHead(nn.Module):
    def __init__(self, dim_in=200, feat_dim=128):
        super(ProjHead, self).__init__()
        self.head = nn.Linear(dim_in, feat_dim)

    def forward(self, x):
        #x = F.normalize(x, dim=1)
        z_proj = self.head(x)
        z_proj = F.normalize(z_proj, dim=1)
        return z_proj



# Define the Wrapper
class Wrapper(nn.Module):
    def __init__(self, CoreModel, ProjLayer):
        super(Wrapper, self).__init__()
        self.CoreModel = CoreModel
        self.ProjLayer = ProjLayer
        #self.LC = LC

    def forward(self, x, return_z=False):
        output, z = self.CoreModel(x, return_z=True)
        z_proj = self.ProjLayer(z)
        #output = self.LC(z)
        if return_z:
            return output, z_proj
        else:
            return output

# Instantiate the components
core_model = CoreModel()
proj_head = ProjHead(200, 128)
#linear_classifier = LinearClassifier(128, 2)

# Combine them in the Wrapper
combined_model = Wrapper(core_model, proj_head)

# Example usage
x = torch.randn(2, 10000)  # Example input
output, z_proj = combined_model(x,return_z=True)
print(output.shape)
print(z_proj.shape)

In [ ]:
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

combined_model = Wrapper(core_model, proj_head).to(device)
combined_model.load_state_dict(torch.load('model_AT_max%20(4).pth', map_location=torch.device(device)))

In [ ]:
def pgd_min22(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', confidence=0.1, initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    #x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        #loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))

        batch_size = x_var.shape[0]
        real = y_model[torch.arange(batch_size), y.view(-1).long()]
        other = y_model[torch.arange(batch_size), (y.view(-1).long()- 1)]
        loss = torch.clamp((real - other + confidence), min=0.0)
        #loss = (real - other)


        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        grad4insertion = (gradients >= 0) *(x_var < 1.)* gradients
        #grad4removal = (gradients < 0) * (x_var > 0.) * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    #x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    #loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    #replace_flag = (loss_adv > loss_natural).squeeze()
    #x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 10000
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

            #print(outputs)
            print(len(mal_y_batch))
            print((y_pred == 0).sum().item())
            #print((y_pred == 0))
            #print('mean difference ',(torch.abs(pertb_mal_x - mal_x_batch)).sum()/len(mal_y_batch))
            #print((torch.abs(pertb_mal_x[y_pred == 0] - mal_x_batch[y_pred == 0])).sum())
            #print((torch.abs(pertb_mal_x - mal_x_batch)).sum(1))

            #print('***************************')


    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

    return pertb_mal_x

In [ ]:
def sum_of_top_n_differences(adv_rounded, mal_x_batch, y_pred, n):
    """
    Calculate the sum of the top n smallest differences between adv_rounded and mal_x_batch
    for samples where y_pred == 0.

    Parameters:
    adv_rounded (torch.Tensor): The rounded adversarial examples tensor.
    mal_x_batch (torch.Tensor): The original malicious examples tensor.
    y_pred (torch.Tensor): The prediction tensor.
    n (int): The number of top differences to sum.

    Returns:
    torch.Tensor: The sum of the top n smallest differences.
    """
    # Step 1: Calculate the total differences
    total_difference = torch.abs(adv_rounded[y_pred == 0] - mal_x_batch[y_pred == 0]).sum(1)

    # Step 2: Sort the differences in ascending order
    sorted_differences, _ = torch.sort(total_difference)

    # Step 3: Select the top n least differences
    top_n_least_differences = sorted_differences[:n]

    # Step 4: Sum these n differences
    sum_top_n_least_differences = top_n_least_differences.sum()

    return sum_top_n_least_differences

In [ ]:
def evaluate_non_rounded_adversarial_examples(mal_x_batch, mal_y_batch, adv, combined_model):
    """
    Evaluate the model's performance on non-rounded adversarial examples.

    Parameters:
    - mal_x_batch: Original malware input batch
    - mal_y_batch: Ground truth labels for malware batch
    - adv: Adversarial examples generated from mal_x_batch
    - combined_model: The trained model being evaluated

    Returns:
    - A dictionary containing evaluation metrics
    """

    # Set the model to evaluation mode
    combined_model.eval()

    # Get the model's predictions on adversarial examples
    outputs = combined_model(adv)
    y_pred = outputs.argmax(dim=1)

    # Calculate metrics
    num_malwares = len(mal_y_batch)
    num_successful_attacks = (y_pred == 0).sum().item()
    total_difference = (torch.abs(adv[y_pred == 0] - mal_x_batch[y_pred == 0])).sum().item()
    total_difference_TOP100 = sum_of_top_n_differences(adv, mal_x_batch, y_pred, 100)
    accuracy = (y_pred == 1).sum().item() / num_malwares * 100

    # Print the evaluation results
    print(f"Evaluation Results on NOT_Rounded Adversarial Examples")
    print(f"-----------------------------------------------")
    print(f"Number of malware samples: {num_malwares}")
    print(f"Number of successful attacks: {num_successful_attacks}")
    print(f"Total difference (L1 norm) between successful adversarial examples and original: {total_difference}")
    print(f"Total L1 norm difference - TOP 100 (successful adversarial examples vs. original): {total_difference_TOP100}")
    print(f"Accuracy of the model on malwares under attack: {accuracy:.2f}%")

    # Return results as a dictionary
    return {
        "num_malwares": num_malwares,
        "num_successful_attacks": num_successful_attacks,
        "total_difference": total_difference,
        "total_difference_TOP100": total_difference_TOP100,
        "accuracy": accuracy
    }

In [ ]:
def evaluate_rounded_adversarial_examples(mal_x_batch, mal_y_batch, adv, combined_model, removal_array):
    """
    Evaluate the model's performance on rounded adversarial examples.

    Parameters:
    - mal_x_batch: Original malware input batch
    - mal_y_batch: Ground truth labels for malware batch
    - adv: Adversarial examples generated from mal_x_batch
    - combined_model: The trained model being evaluated
    - removal_array: Array used in the final rounding step

    Returns:
    - A dictionary containing evaluation metrics
    """

    # Set the model to evaluation mode
    combined_model.eval()

    # Apply the final rounding solution to adversarial examples
    adv_rounded = final_rounding(mal_x_batch, adv, combined_model, removal_array)

    # Get the model's predictions on the rounded adversarial examples
    outputs = combined_model(adv_rounded)
    y_pred = outputs.argmax(dim=1)

    # Calculate evaluation metrics
    num_malwares = len(mal_y_batch)
    num_successful_attacks = (y_pred == 0).sum().item()
    total_difference = (torch.abs(adv_rounded[y_pred == 0] - mal_x_batch[y_pred == 0])).sum().item()
    total_difference_TOP100 = sum_of_top_n_differences(adv_rounded, mal_x_batch, y_pred, 100)
    total_difference_TOP150 = sum_of_top_n_differences(adv_rounded, mal_x_batch, y_pred, 150)
    total_difference_TOP200 = sum_of_top_n_differences(adv_rounded, mal_x_batch, y_pred, 200)
    accuracy = (y_pred == 1).sum().item() / num_malwares * 100

    # Print the evaluation results
    print(f"Evaluation Results on Rounded Adversarial Examples")
    print(f"-----------------------------------------------")
    print(f"Number of malware samples: {num_malwares}")
    print(f"Number of successful attacks: {num_successful_attacks}")
    print(f"Total L1 norm difference (successful adversarial examples vs. original): {total_difference}")
    print(f"Total L1 norm difference - TOP 100 (successful adversarial examples vs. original): {total_difference_TOP100}")
    print(f"Total L1 norm difference - TOP 150 (successful adversarial examples vs. original): {total_difference_TOP150}")
    print(f"Total L1 norm difference - TOP 200 (successful adversarial examples vs. original): {total_difference_TOP200}")
    print(f"Accuracy of the model on malware under attack: {accuracy:.2f}%")

    # Return results as a dictionary
    return {
        "adv_rounded": adv_rounded,
        "num_malwares": num_malwares,
        "num_successful_attacks": num_successful_attacks,
        "total_difference": total_difference,
        "total_difference_TOP100": total_difference_TOP100,
        "total_difference_TOP150": total_difference_TOP150,
        "total_difference_TOP200": total_difference_TOP200,
        "accuracy": accuracy
    }

In [ ]:
def ensure_fixed_elements_unchanged(mal_x_batch, adv, adv_rounded, removal_array):
    """
    Ensure that fixed elements are not changed and calculate differences between adversarial examples and the original batch.

    Parameters:
    - mal_x_batch: Original malware input batch
    - adv: Adversarial examples generated from mal_x_batch
    - adv_rounded: Rounded adversarial examples
    - removal_array: Array indicating the fixed features

    Returns:
    - A dictionary containing differences for fixed and non-fixed features
    """

    # Ensure `removal_array` is expanded to match the batch size
    expanded_removal_array = removal_array.expand(mal_x_batch.shape[0], -1)

    # Create a mask where non-fixed features are set to 1
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - mal_x_batch.to(torch.uint8))

    # Calculate indices where the mask is 0 (fixed) or 1 (non-fixed)
    mask_fixed = non_fixed_features_mask == 0
    mask_non_fixed = non_fixed_features_mask == 1

    # Calculate the differences for fixed and non-fixed features
    diff_adv_fixed = (torch.abs(adv - mal_x_batch) * mask_fixed).sum().item()
    diff_adv_non_fixed = (torch.abs(adv - mal_x_batch) * mask_non_fixed).sum().item()
    diff_adv_rounded_fixed = (torch.abs(adv_rounded - mal_x_batch) * mask_fixed).sum().item()
    diff_adv_rounded_non_fixed = (torch.abs(adv_rounded - mal_x_batch) * mask_non_fixed).sum().item()

    # Print the differences
    print(f"Differences when mask is 0 (fixed features):")
    print(f"  Adv vs. Original: {diff_adv_fixed}")
    print(f"  Rounded Adv vs. Original: {diff_adv_rounded_fixed}")
    print(f"Differences when mask is 1 (non-fixed features):")
    print(f"  Adv vs. Original: {diff_adv_non_fixed}")
    print(f"  Rounded Adv vs. Original: {diff_adv_rounded_non_fixed}")

    # Return the results as a dictionary
    return {
        "diff_adv_fixed": diff_adv_fixed,
        "diff_adv_non_fixed": diff_adv_non_fixed,
        "diff_adv_rounded_fixed": diff_adv_rounded_fixed,
        "diff_adv_rounded_non_fixed": diff_adv_rounded_non_fixed
    }

In [ ]:
def compare_round(mal_x_batch,newimg, model,removal_array):

    adv_rounded = final_rounding(mal_x_batch, newimg, model,removal_array)
    #l0dist = torch.sum((mal_x_batch - newimg)!=0, dim=1)
    l0dist = torch.sum(torch.abs(adv_rounded - mal_x_batch), dim=1)
    logits = model(adv_rounded)
    output = torch.argmax(logits, dim=1)

    return output == 0, l0dist

In [ ]:
def adv_predict_rounding(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            adv_rounded = final_rounding(mal_x_batch, pertb_mal_x, model,removal_array)

            outputs = model(adv_rounded)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

            #print(outputs)
            print(len(mal_y_batch))
            print((y_pred == 0).sum().item())
            #print((y_pred == 0))
            #print('mean difference ',(torch.abs(pertb_mal_x - mal_x_batch)).sum()/len(mal_y_batch))
            #print((torch.abs(pertb_mal_x[y_pred == 0] - mal_x_batch[y_pred == 0])).sum())
            #print((torch.abs(pertb_mal_x - mal_x_batch)).sum(1))

            #print('***************************')


    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

    return pertb_mal_x

In [ ]:
def final_rounding(oimgs, newimg, model, removal_array):
    # Expand removal array to match the batch size
    non_fixed_features_mask = torch.bitwise_or(removal_array.expand(oimgs.shape[0], -1), 1 - oimgs.to(torch.uint8)).to(device)

    # Ensure all tensors are on the correct device
    oimgs, perturbed = oimgs.to(device), newimg.to(device)

    # Set the model to evaluation mode
    model.eval()

    # Calculate predictions and significant changes efficiently using GPU
    with torch.no_grad():
        predictions = model(oimgs).argmax(dim=1).to(device)
        active_mask = predictions == 1  # Efficiently create active mask on GPU
        total_change = (perturbed - oimgs).abs().to(device)  # Calculate absolute difference on GPU
        significant_mask = total_change > 0.1  # Determine significant changes on GPU
        sorted_idx = total_change.argsort(dim=1, descending=True)  # Sort on GPU
        changes = (perturbed > oimgs).float().to(device)

    # Copy original images to apply final rounding
    final = oimgs.clone().to(device)

    # Iterate up to the maximum number of significant changes
    # practicaly we dont have perturbs more than 120 !!
    max_significant_changes = min(significant_mask.sum(dim=1).max().item(), 120)

    for i in range(max_significant_changes):
        # Safeguard against empty active mask
        if not active_mask.any():
            break

        # Get the indices of active samples
        active_indices = torch.where(active_mask)[0].to(device)

        # Get the relevant indices for current change
        indices = sorted_idx[active_indices, i]

        # since max_significant_changes is relatively large for some samples, possible that zero changes which belong to fixed_features get updated
        final[active_indices, indices] = torch.where(non_fixed_features_mask[active_indices, indices],
                                                     changes[active_indices, indices], final[active_indices, indices])

        # Update the active_mask based on the model's prediction
        with torch.no_grad():
            active_mask[active_indices] = (model(final[active_indices]).argmax(dim=1) == 1)

    return final

In [ ]:
for x_test, y_test in test_loader:
    x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
    mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
    break

## Step 2 — The learning-rate functions

# LR functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define parameters
lr_start = 1e-3
lr_end = 0.9
T = 100  # Total number of iterations

# Define the learning rate schedules

# Linear schedule
def lr_linear(t, T, lr_start, lr_end):
    return lr_start + t * (lr_end - lr_start) / T

# Exponential schedule
def lr_exponential(t, T, lr_start, lr_end):
    return lr_start * (lr_end / lr_start) ** (t / T)

def lr_average(t, T, lr_start, lr_end):
    lr_linear_val = lr_linear(t, T, lr_start, lr_end)
    lr_exponential_val = lr_exponential(t, T, lr_start, lr_end)
    return 0.5 * (lr_linear_val + lr_exponential_val)

# Cosine schedule
def lr_cosine(t, T, lr_start, lr_end):
    return lr_end + (lr_start - lr_end) * 0.5 * (1 + np.cos(np.pi * t / T))

def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))


# Generate values for plotting
iterations = np.arange(0, T+1)
lr_linear_values = [lr_linear(t, T, lr_start, lr_end) for t in iterations]
lr_exponential_values = [lr_exponential(t, T, lr_start, lr_end) for t in iterations]
lr_average_values = [lr_average(t, T, lr_start, lr_end) for t in iterations]
lr_cosine_values = [lr_cosine(t, T, lr_start, lr_end) for t in iterations]
lr_cosine_half_values = [lr_cosine_half(t, T, lr_start, lr_end) for t in iterations]



# Plot the learning rate schedules
plt.figure(figsize=(10, 6))
plt.plot(iterations, lr_linear_values, label="Linear Schedule", color='blue', linestyle='--')
plt.plot(iterations, lr_exponential_values, label="Exponential Schedule", color='green', linestyle='-.')
plt.plot(iterations, lr_average_values, label="Average Schedule", color='purple', linestyle=':')
plt.plot(iterations, lr_cosine_values, label="Cosine Schedule", color='red', linestyle='-')
plt.plot(iterations, lr_cosine_half_values, label="Cosine Schedule Half", color='orange', linestyle='-')


plt.title('Learning Rate Schedules')
plt.xlabel('Iteration')
plt.ylabel('Learning Rate')
plt.legend()
plt.grid(True)
plt.show()

## Step 3 — The old sine function, and the first grids

# Old sin function:
(torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim.lr_scheduler import LambdaLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.
    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

# Define the learning rate schedules (same as earlier)
def lr_linear(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start + t * (lr_end - lr_start) / T

def lr_exponential(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start * (lr_end / lr_start) ** (t / T)

def lr_cosine(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_end + (lr_start - lr_end) * 0.5 * (1 + torch.cos(torch.tensor(np.pi * t / T)))

def lr_average(t, T, lr_start=1e-3, lr_end=0.9):
    return 0.5 * (lr_linear(t, T, lr_start, lr_end) + lr_exponential(t, T, lr_start, lr_end))

def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))



def CW_tanh_test(model, imgs, y, removal_array, device='cuda', const=100.0, LR_function=lr_exponential,
                 max_lr=2., min_lr=1e-4,update_step=1000, first_phase_iters=20000,second_phase_iters = 10000 , confidence=0.1, eps=1e-4):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - max_lr: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform.
    - confidence: Confidence parameter for adversarial loss computation.
    """
    # Preparing tensors
    imgs_tensor_original = imgs.clone().detach().to(dtype=torch.float32, device=device)
    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)

    # Create masks for 0s and 1s
    zero_mask = (imgs_tensor == 0)  # Mask where values are 0
    one_mask = (imgs_tensor == 1)   # Mask where values are 1

    # Add eps to 0s and subtract eps from 1s
    imgs_tensor[zero_mask] += eps
    imgs_tensor[one_mask] -= eps

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
    optimizer = torch.optim.Adam([modifier], lr=min_lr)


    def grad_hook(grad):
        return grad * non_fixed_features_mask

    modifier.register_hook(grad_hook)

    # Initialize tracking variables
    batch_size = imgs.shape[0]
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()
    o_best_modifier = torch.zeros_like(modifier).detach()  # Best modifier
    o_best_unsuccessfull_modifier = torch.zeros_like(modifier).detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    # Total number of iterations
    total_iters = first_phase_iters + second_phase_iters

    for iteration in range(total_iters):

        # Phase 2: Fixed learning rate of 1e-3 (iterations 15,000 to 24,999)
        if iteration == first_phase_iters:
            print('')
            output = model(o_bestattack)
            final_success = compare(output)
            print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.4f}\t success Loss = {loss[final_success].sum().item():.4f} \t success l2dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
            print('')
            final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
            final_modifier = torch.where(final_success.unsqueeze(-1), o_best_modifier, o_best_unsuccessfull_modifier)
            results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_output, combined_model, removal_array)
            print('')

            modifier = torch.tensor(final_modifier.clone().detach(), requires_grad=True).to(device)
            optimizer = torch.optim.Adam([modifier], lr=1e-3)  # Use lr_min for 1000 iterations
            modifier.register_hook(grad_hook)

        if iteration < first_phase_iters:

            if iteration == 0 or (iteration + 1) % update_step == 0:
                # Select your preferred learning rate schedule
                current_lr = LR_function(iteration, first_phase_iters, min_lr, max_lr)  # You can switch to lr_linear, lr_exponential, or lr_cosine

                # Update the optimizer's learning rate
                for param_group in optimizer.param_groups:
                    param_group['lr'] = current_lr


        optimizer.zero_grad()

        # Attack Step: generate perturbed image
        modified_input = imgs_tensor + modifier
        newimg = sin_func_torch(modified_input)
        output = model(newimg)

        l2dist = torch.sum((newimg - imgs_tensor_original) ** 2, dim=1)

        real = output[:, 0]
        other = output[:, 1]

        loss1 = torch.clamp((other - real + confidence), min=0.0)
        loss = (const * loss1) + l2dist

        total_loss = torch.sum(loss)
        total_loss.backward()
        optimizer.step()


        with torch.no_grad():
            success_mask_inloop = compare(output)
            success_mask = torch.logical_or(success_mask_inloop, success_mask)

            update_mask = (l2dist < o_bestl2) & success_mask_inloop
            o_bestl2 = torch.where(update_mask, l2dist, o_bestl2)
            o_bestattack = torch.where(update_mask.unsqueeze(-1), newimg, o_bestattack)
            o_best_modifier = torch.where(update_mask.unsqueeze(-1), modifier.detach(), o_best_modifier)

            improved_mask = (loss1 < o_best_loss1)
            o_best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
            o_best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)
            o_best_unsuccessfull_modifier = torch.where(improved_mask.unsqueeze(-1), modifier.detach(), o_best_unsuccessfull_modifier)

            if iteration == 0 or (iteration + 1) % 500 == 0:
                current_lr = optimizer.param_groups[0]['lr']
                current_momentum, current_momentum2 = optimizer.param_groups[0]['betas']
                print(f"Iteration {iteration + 1:5}:\t Current LR = {current_lr:.4f} \t Current momentum = {current_momentum:.2f} \t "
                      f"Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t "
                      f"Loss = {total_loss.item():.4f} \t l2dist = {torch.mean(o_bestl2[success_mask]).item():.4f}")

    print('')
    output = model(o_bestattack)
    final_success = compare(output)
    print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.4f}\t "
          f"success Loss = {loss[final_success].sum().item():.4f} \t success l2dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
    print('')

    final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_output, combined_model, removal_array)
    print('')

In [ ]:
import itertools

# Define hyperparameter grid
param_grid = {
    'max_lr': [0.3,0.6,0.9,1.2,1.5],
    'LR_function': [lr_linear,lr_exponential,lr_average,lr_cosine],
}


for params in itertools.product(*param_grid.values()):
    # Check if this set of parameters gives the best result
    print('**********************************************************************************************************************************************')
    max_lr, LR_function = params
    print(f"Params:\t max_lr={max_lr}\t LR_function={LR_function.__name__}")
    print('-------------------------------')
    CW_tanh_test(combined_model, mal_x_batch, mal_y_batch, removal_array, device='cuda', const=100.0, LR_function=LR_function,
                 max_lr=max_lr, min_lr=1e-3, update_step=1, first_phase_iters=20000,second_phase_iters = 5000 , confidence=0.1, eps=1e-4)

In [ ]:
import itertools

# Define hyperparameter grid
param_grid = {
    'max_lr': [0.6,0.9,1.2,1.5],
    'LR_function': [lr_linear,lr_exponential,lr_average,lr_cosine,lr_cosine_half],
}


for params in itertools.product(*param_grid.values()):
    # Check if this set of parameters gives the best result
    print('**********************************************************************************************************************************************')
    max_lr, LR_function = params
    print(f"Params:\t max_lr={max_lr}\t LR_function={LR_function.__name__}")
    print('-------------------------------')
    CW_tanh_test(combined_model, mal_x_batch, mal_y_batch, removal_array, device='cuda', const=100.0, LR_function=LR_function,
                 max_lr=max_lr, min_lr=1e-3, update_step=500, first_phase_iters=20000,second_phase_iters = 5000 , confidence=0.1, eps=1e-4)

## Updates of modifiers

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim.lr_scheduler import LambdaLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.
    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0


def cos_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.
    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (1 - torch.cos(input_tensor * torch.pi)) / 2.0


def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))



def CW_tanh_test(model, imgs, y, removal_array, device='cuda', const=100.0, function=sin_func_torch,
                 max_lr=2., min_lr=1e-4,update_step=1000, first_phase_iters=20000,second_phase_iters = 10000 , confidence=0.1, eps=1e-4, betas=(0.9,0.99)):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - max_lr: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform.
    - confidence: Confidence parameter for adversarial loss computation.
    """
    # Preparing tensors
    imgs_tensor_original = imgs.clone().detach().to(dtype=torch.float32, device=device)
    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)

    # Create masks for 0s and 1s
    zero_mask = (imgs_tensor == 0)  # Mask where values are 0
    one_mask = (imgs_tensor == 1)   # Mask where values are 1

    # Add eps to 0s and subtract eps from 1s
    imgs_tensor[zero_mask] += eps
    imgs_tensor[one_mask] -= eps

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
    previous_modifier = modifier.clone().detach()
    optimizer = torch.optim.Adam([modifier], lr=min_lr, betas=betas)


    def grad_hook(grad):
        return grad * non_fixed_features_mask

    modifier.register_hook(grad_hook)

    # Initialize tracking variables
    batch_size = imgs.shape[0]
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()
    o_best_modifier = torch.zeros_like(modifier).detach()  # Best modifier
    o_best_unsuccessfull_modifier = torch.zeros_like(modifier).detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    # Total number of iterations
    total_iters = first_phase_iters + second_phase_iters

    for iteration in range(total_iters):

        # Phase 2: Fixed learning rate of 1e-3 (iterations 15,000 to 24,999)
        if iteration == first_phase_iters:
            print('')
            output = model(o_bestattack)
            final_success = compare(output)
            print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.4f}\t success Loss = {loss[final_success].sum().item():.4f} \t success l2dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
            print('')
            final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
            final_modifier = torch.where(final_success.unsqueeze(-1), o_best_modifier, o_best_unsuccessfull_modifier)
            results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_output, combined_model, removal_array)
            print('')

            modifier = torch.tensor(final_modifier.clone().detach(), requires_grad=True).to(device)
            optimizer = torch.optim.Adam([modifier], lr=1e-3)  # Use lr_min for 1000 iterations
            modifier.register_hook(grad_hook)

        if iteration < first_phase_iters:

            if iteration == 0 or (iteration + 1) % update_step == 0:
                # Select your preferred learning rate schedule
                current_lr = lr_cosine_half(iteration, first_phase_iters, min_lr, max_lr)  # You can switch to lr_linear, lr_exponential, or lr_cosine

                # Update the optimizer's learning rate
                for param_group in optimizer.param_groups:
                    param_group['lr'] = current_lr


        optimizer.zero_grad()

        # Attack Step: generate perturbed image
        modified_input = imgs_tensor + modifier
        newimg = function(modified_input)
        output = model(newimg)

        l2dist = torch.sum((newimg - imgs_tensor_original) ** 2, dim=1)

        real = output[:, 0]
        other = output[:, 1]

        loss1 = torch.clamp((other - real + confidence), min=0.0)
        loss = (const * loss1) + l2dist

        total_loss = torch.sum(loss)
        total_loss.backward()

        '''
        # Modify gradients to zero out updates for samples exceeding the threshold
        with torch.no_grad():
            grad_mask = continue_update_mask.unsqueeze(-1).float()
            modifier.grad *= grad_mask  # Only apply gradients for samples that still need updating'''

        optimizer.step()

        '''
        # Compute the update from the previous iteration (modifier - previous_modifier)
        with torch.no_grad():
            updates = torch.abs(modifier - previous_modifier)
            sample_max_updates = updates.max(dim=1).values

            # For samples where sample_max_updates > 0.1, stop further updates
            continue_update_mask = continue_update_mask & (sample_max_updates <= 0.1)
            print(f"Samples to continue updating: {(continue_update_mask).sum().item()} out of {batch_size}")

        # Update the previous_modifier to the current modifier for the next iteration
        previous_modifier = modifier.clone().detach()'''

        optimizer.step()


        with torch.no_grad():
            success_mask_inloop = compare(output)
            success_mask = torch.logical_or(success_mask_inloop, success_mask)

            update_mask = (l2dist < o_bestl2) & success_mask_inloop
            o_bestl2 = torch.where(update_mask, l2dist, o_bestl2)
            o_bestattack = torch.where(update_mask.unsqueeze(-1), newimg, o_bestattack)
            o_best_modifier = torch.where(update_mask.unsqueeze(-1), modifier.detach(), o_best_modifier)

            improved_mask = (loss1 < o_best_loss1)
            o_best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
            o_best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)
            o_best_unsuccessfull_modifier = torch.where(improved_mask.unsqueeze(-1), modifier.detach(), o_best_unsuccessfull_modifier)

            if iteration == 0 or (iteration + 1) % 500 == 0 or (iteration + 1) % 500 == 1:
                current_lr = optimizer.param_groups[0]['lr']
                current_momentum, current_momentum2 = optimizer.param_groups[0]['betas']

                print(f"Iteration {iteration + 1:5}:\t Current LR = {current_lr:.4f} \t Current momentum = {current_momentum:.2f} \t "
                      f"Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t "
                      f"Loss = {total_loss.item():.4f} \t l2dist = {torch.mean(o_bestl2[success_mask]).item():.4f}")

                updates = torch.abs(modifier - previous_modifier)   # Difference between current and previous modifier
                # Find the maximum update for each sample
                sample_max_updates = updates.max(dim=1).values   # Max update of each sample

                print(f"Iteration {iteration + 1:5}:\t max_updates > 0.1 = {(sample_max_updates > 0.1).sum():4} \t max_updates > 0.2 = {(sample_max_updates > 0.2).sum():4} \t "
                      f"max_updates > 0.4 = {(sample_max_updates > 0.4).sum():4} \t max_updates > 0.6 = {(sample_max_updates > 0.6).sum():4} \t "
                      f"max_updates > 0.8 = {(sample_max_updates > 0.8).sum():4} \t lmax_updates > 1. = {(sample_max_updates > 1.).sum():4}")
                print('----------')

        # Update the previous_modifier to the current modifier for the next iteration
        previous_modifier = modifier.clone().detach()


    print('')
    output = model(o_bestattack)
    final_success = compare(output)
    print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.4f}\t "
          f"success Loss = {loss[final_success].sum().item():.4f} \t success l2dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
    print('')

    final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_output, combined_model, removal_array)
    print('')

In [ ]:
CW_tanh_test(combined_model, mal_x_batch, mal_y_batch, removal_array, device=device, const=100.0, function=sin_func_torch,
                 max_lr=0.6, min_lr=1e-3, update_step=500, first_phase_iters=20000,second_phase_iters=5000, confidence=0.1, eps=1e-4)

## Rounded end of the loop - all const

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim.lr_scheduler import LambdaLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.
    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0


# Define the learning rate schedules (same as earlier)
def lr_linear(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start + t * (lr_end - lr_start) / T

def lr_exponential(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start * (lr_end / lr_start) ** (t / T)

def lr_cosine(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_end + (lr_start - lr_end) * 0.5 * (1 + torch.cos(torch.tensor(np.pi * t / T)))

def lr_average(t, T, lr_start=1e-3, lr_end=0.9):
    return 0.5 * (lr_linear(t, T, lr_start, lr_end) + lr_exponential(t, T, lr_start, lr_end))

def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))



def CW(model, imgs, y, removal_array, device='cuda', initial_const = 1., binary_search_steps=8, cons_increase_factor=10., LR_function=lr_exponential,
                 max_lr=2., min_lr=1e-4,update_step=1000, first_phase_iters=20000,second_phase_iters = 10000 , confidence=0.1, eps=1e-4):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - max_lr: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform.
    - confidence: Confidence parameter for adversarial loss computation.
    """

    def grad_hook(grad):
        return grad * non_fixed_features_mask

    # Preparing tensors
    imgs_tensor_original = imgs.clone().detach().to(dtype=torch.float32, device=device)
    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)
    batch_size = imgs.shape[0]

    # Create masks for 0s and 1s
    zero_mask = (imgs_tensor == 0)  # Mask where values are 0
    one_mask = (imgs_tensor == 1)   # Mask where values are 1

    # Add eps to 0s and subtract eps from 1s
    imgs_tensor[zero_mask] += eps
    imgs_tensor[one_mask] -= eps

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))


    # Initialize tracking variables
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    # Total number of iterations
    total_iters = first_phase_iters + second_phase_iters

    for outer_step in range(binary_search_steps):
        print('------------------------------------------------------------------------------------------------------------------------------------')
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=min_lr)
        modifier.register_hook(grad_hook)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestattack = imgs.clone().detach()
        best_modifier = torch.zeros_like(modifier).detach()  # Best modifier
        best_unsuccessfull_modifier = torch.zeros_like(modifier).detach()

        best_loss1 = torch.full((batch_size,), float('inf'), device=device)
        best_unsuccessfull_attack = imgs.clone().detach()
        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(total_iters):

            # Phase 2: Fixed learning rate of 1e-3 (iterations 15,000 to 24,999)
            if iteration == first_phase_iters:
                print("************ second phase *************")
                output = model(o_bestattack)
                final_success = compare(output)
                final_output = torch.where(final_success.unsqueeze(-1), bestattack, best_unsuccessfull_attack)
                final_modifier = torch.where(final_success.unsqueeze(-1), best_modifier, best_unsuccessfull_modifier)

                modifier = torch.tensor(final_modifier.clone().detach(), requires_grad=True).to(device)
                optimizer = torch.optim.Adam([modifier], lr=1e-3)  # Use lr_min for 1000 iterations
                modifier.register_hook(grad_hook)

            if iteration < first_phase_iters:
                if iteration == 0 or (iteration + 1) % update_step == 0:
                    # Select your preferred learning rate schedule
                    current_lr = LR_function(iteration, first_phase_iters, min_lr, max_lr)  # You can switch to lr_linear, lr_exponential, or lr_cosine

                    # Update the optimizer's learning rate
                    for param_group in optimizer.param_groups:
                        param_group['lr'] = current_lr

            optimizer.zero_grad()

            # Attack Step: generate perturbed image
            modified_input = imgs_tensor + modifier
            newimg = sin_func_torch(modified_input)
            output = model(newimg)

            l2dist = torch.sum((newimg - imgs_tensor_original) ** 2, dim=1)

            real = output[:, 0]
            other = output[:, 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            loss = (CONST * loss1) + l2dist

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()

            with torch.no_grad():
                success_mask_inloop = compare(output)
                success_mask = torch.logical_or(success_mask_inloop, success_mask)
                o_success_mask  = torch.logical_or(o_success_mask, success_mask_inloop)

                update_mask = (l2dist < bestl2) & success_mask_inloop
                bestl2 = torch.where(update_mask, l2dist, bestl2)
                bestattack = torch.where(update_mask.unsqueeze(-1), newimg, bestattack)
                best_modifier = torch.where(update_mask.unsqueeze(-1), modifier.detach(), best_modifier)

                improved_mask = (loss1 < best_loss1)
                best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
                best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, best_unsuccessfull_attack)
                best_unsuccessfull_modifier = torch.where(improved_mask.unsqueeze(-1), modifier.detach(), best_unsuccessfull_modifier)

                update_mask2 = (l2dist < o_bestl2) & success_mask_inloop
                o_bestl2 = torch.where(update_mask2, l2dist, o_bestl2)
                o_bestattack = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack)

                improved_mask2 = (loss1 < o_best_loss1)
                o_best_loss1 = torch.where(improved_mask2, loss1, o_best_loss1)
                o_best_unsuccessfull_attack = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)


                if iteration == 0 or (iteration + 1) % 500 == 0:
                    current_lr = optimizer.param_groups[0]['lr']
                    current_momentum, current_momentum2 = optimizer.param_groups[0]['betas']
                    print(f"Iteration {iteration + 1:5}:\t Current LR = {current_lr:.4f} \t Current momentum = {current_momentum:.2f} \t Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t Loss = {total_loss.item():.4f} \t l2dist = {torch.mean(bestl2[success_mask]).item():.4f}")

                    print(f"Best results until now: \t success = {o_success_mask.sum().item()} \t l2dist = {torch.mean(o_bestl2[o_success_mask]).item():.4f}")

        failure_mask = ~success_mask
        upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
        lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

        upper_limit_mask = upper_bound < 1e9
        CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
        CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor



    print('')
    output = model(o_bestattack)
    final_success = compare(output)
    print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.4f}\t success Loss = {loss[final_success].sum().item():.4f} \t success l2dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
    print('')

    final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_output, combined_model, removal_array)
    print('')

In [ ]:
CW(combined_model, mal_x_batch, mal_y_batch, removal_array, device='cuda', initial_const = 1., binary_search_steps=9, cons_increase_factor=10., LR_function=lr_cosine_half,
                 max_lr=0.9, min_lr=1e-3,update_step=500, first_phase_iters=20000,second_phase_iters = 5000 , confidence=0.1, eps=1e-4)

In [ ]:
CW(combined_model, mal_x_batch, mal_y_batch, removal_array, device='cuda', initial_const = 1., binary_search_steps=9, cons_increase_factor=10., LR_function=lr_cosine_half,
                 max_lr=0.9, min_lr=1e-4,update_step=500, first_phase_iters=20000,second_phase_iters = 5000 , confidence=0.1, eps=1e-4)

## Step 4 — The new sine function

# New sin function:
newimg = torch.sin(modified_input) * 0.5 + 0.5

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim.lr_scheduler import LambdaLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0


def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))



def CW_tanh_test(model, imgs, y, removal_array, device='cuda', const=100.0, function=sin_func_torch,
                 max_lr=2., min_lr=1e-4,update_step=1000, first_phase_iters=20000,second_phase_iters = 10000 , confidence=0.1, eps=1e-4, betas=(0.9,0.99)):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - max_lr: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform.
    - confidence: Confidence parameter for adversarial loss computation.
    """
    # Preparing tensors
    imgs_tensor_original = imgs.clone().detach().to(dtype=torch.float32, device=device)
    imgs_tensor = torch.where(imgs.clone().detach().to(torch.uint8), torch.pi * 0.5, -torch.pi * 0.5).to(device)
    labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)

    # Create masks for 0s and 1s
    zero_mask = (imgs == 0)  # Mask where values are 0
    one_mask = (imgs == 1)   # Mask where values are 1

    # Add eps to 0s and subtract eps from 1s
    imgs_tensor[zero_mask] += eps
    imgs_tensor[one_mask] -= eps

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
    previous_modifier = modifier.clone().detach()
    optimizer = torch.optim.Adam([modifier], lr=min_lr, betas=betas)


    def grad_hook(grad):
        return grad * non_fixed_features_mask

    modifier.register_hook(grad_hook)

    # Initialize tracking variables
    batch_size = imgs.shape[0]
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()
    o_best_modifier = torch.zeros_like(modifier).detach()  # Best modifier
    o_best_unsuccessfull_modifier = torch.zeros_like(modifier).detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    # Total number of iterations
    total_iters = first_phase_iters + second_phase_iters

    for iteration in range(total_iters):

        # Phase 2: Fixed learning rate of 1e-3 (iterations 15,000 to 24,999)
        if iteration == first_phase_iters:
            print('')
            output = model(o_bestattack)
            final_success = compare(output)
            print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.4f}\t success Loss = {loss[final_success].sum().item():.4f} \t success l2dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
            print('')
            final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
            final_modifier = torch.where(final_success.unsqueeze(-1), o_best_modifier, o_best_unsuccessfull_modifier)
            results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_output, combined_model, removal_array)
            print('')

            modifier = torch.tensor(final_modifier.clone().detach(), requires_grad=True).to(device)
            optimizer = torch.optim.Adam([modifier], lr=1e-3)  # Use lr_min for 1000 iterations
            modifier.register_hook(grad_hook)

        if iteration < first_phase_iters:

            if iteration == 0 or (iteration + 1) % update_step == 0:
                # Select your preferred learning rate schedule
                current_lr = lr_cosine_half(iteration, first_phase_iters, min_lr, max_lr)  # You can switch to lr_linear, lr_exponential, or lr_cosine

                # Update the optimizer's learning rate
                for param_group in optimizer.param_groups:
                    param_group['lr'] = current_lr


        optimizer.zero_grad()

        # Attack Step: generate perturbed image
        modified_input = imgs_tensor + modifier
        newimg = torch.sin(modified_input) * 0.5 + 0.5
        output = model(newimg)

        l2dist = torch.sum((newimg - imgs_tensor_original) ** 2, dim=1)

        real = output[:, 0]
        other = output[:, 1]

        loss1 = torch.clamp((other - real + confidence), min=0.0)
        loss = (const * loss1) + l2dist

        total_loss = torch.sum(loss)
        total_loss.backward()
        optimizer.step()

        with torch.no_grad():
            success_mask_inloop = compare(output)
            success_mask = torch.logical_or(success_mask_inloop, success_mask)

            update_mask = (l2dist < o_bestl2) & success_mask_inloop
            o_bestl2 = torch.where(update_mask, l2dist, o_bestl2)
            o_bestattack = torch.where(update_mask.unsqueeze(-1), newimg, o_bestattack)
            o_best_modifier = torch.where(update_mask.unsqueeze(-1), modifier.detach(), o_best_modifier)

            improved_mask = (loss1 < o_best_loss1)
            o_best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
            o_best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)
            o_best_unsuccessfull_modifier = torch.where(improved_mask.unsqueeze(-1), modifier.detach(), o_best_unsuccessfull_modifier)

            if iteration == 0 or (iteration + 1) % 500 == 0:
                current_lr = optimizer.param_groups[0]['lr']
                current_momentum, current_momentum2 = optimizer.param_groups[0]['betas']
                print(f"Iteration {iteration + 1:5}:\t Current LR = {current_lr:.4f} \t Current momentum = {current_momentum:.2f} \t "
                      f"Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t "
                      f"Loss = {total_loss.item():.4f} \t l2dist = {torch.mean(o_bestl2[success_mask]).item():.4f}")

    print('')
    output = model(o_bestattack)
    final_success = compare(output)
    print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.4f}\t "
          f"success Loss = {loss[final_success].sum().item():.4f} \t success l2dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
    print('')

    final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_output, combined_model, removal_array)
    print('')

In [ ]:
CW_tanh_test(combined_model, mal_x_batch, mal_y_batch, removal_array, device=device, const=100.0, function=sin_func_torch,
                 max_lr=2.5, min_lr=1e-3, update_step=500, first_phase_iters=20000,second_phase_iters = 5000 , confidence=0.1, eps=1e-4)

## Rounded end of the loop - all const

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim.lr_scheduler import LambdaLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.
    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0


# Define the learning rate schedules (same as earlier)
def lr_linear(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start + t * (lr_end - lr_start) / T

def lr_exponential(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start * (lr_end / lr_start) ** (t / T)

def lr_cosine(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_end + (lr_start - lr_end) * 0.5 * (1 + torch.cos(torch.tensor(np.pi * t / T)))

def lr_average(t, T, lr_start=1e-3, lr_end=0.9):
    return 0.5 * (lr_linear(t, T, lr_start, lr_end) + lr_exponential(t, T, lr_start, lr_end))

def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))



def CW(model, imgs, y, removal_array, device='cuda', initial_const = 1., binary_search_steps=8, cons_increase_factor=10., LR_function=lr_exponential,
                 max_lr=2., min_lr=1e-4,update_step=1000, first_phase_iters=20000,second_phase_iters = 10000 , confidence=0.1, eps=1e-4):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - max_lr: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform.
    - confidence: Confidence parameter for adversarial loss computation.
    """

    def grad_hook(grad):
        return grad * non_fixed_features_mask

    # Preparing tensors
    imgs_tensor_original = imgs.clone().detach().to(dtype=torch.float32, device=device)
    imgs_tensor = torch.where(imgs.clone().detach().to(torch.uint8), torch.pi * 0.5, -torch.pi * 0.5).to(device)
    labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)

    batch_size = imgs.shape[0]

    # Create masks for 0s and 1s
    zero_mask = (imgs == 0)  # Mask where values are 0
    one_mask = (imgs == 1)   # Mask where values are 1

    # Add eps to 0s and subtract eps from 1s
    imgs_tensor[zero_mask] += eps
    imgs_tensor[one_mask] -= eps

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))


    # Initialize tracking variables
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    # Total number of iterations
    total_iters = first_phase_iters + second_phase_iters

    for outer_step in range(binary_search_steps):
        print('------------------------------------------------------------------------------------------------------------------------------------')
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=min_lr)
        modifier.register_hook(grad_hook)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestattack = imgs.clone().detach()
        best_modifier = torch.zeros_like(modifier).detach()  # Best modifier
        best_unsuccessfull_modifier = torch.zeros_like(modifier).detach()

        best_loss1 = torch.full((batch_size,), float('inf'), device=device)
        best_unsuccessfull_attack = imgs.clone().detach()
        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(total_iters):

            # Phase 2: Fixed learning rate of 1e-3 (iterations 15,000 to 24,999)
            if iteration == first_phase_iters:
                print("************ second phase *************")
                output = model(o_bestattack)
                final_success = compare(output)
                final_output = torch.where(final_success.unsqueeze(-1), bestattack, best_unsuccessfull_attack)
                final_modifier = torch.where(final_success.unsqueeze(-1), best_modifier, best_unsuccessfull_modifier)

                modifier = torch.tensor(final_modifier.clone().detach(), requires_grad=True).to(device)
                optimizer = torch.optim.Adam([modifier], lr=1e-3)  # Use lr_min for 1000 iterations
                modifier.register_hook(grad_hook)

            if iteration < first_phase_iters:
                if iteration == 0 or (iteration + 1) % update_step == 0:
                    # Select your preferred learning rate schedule
                    current_lr = LR_function(iteration, first_phase_iters, min_lr, max_lr)  # You can switch to lr_linear, lr_exponential, or lr_cosine

                    # Update the optimizer's learning rate
                    for param_group in optimizer.param_groups:
                        param_group['lr'] = current_lr

            optimizer.zero_grad()

            # Attack Step: generate perturbed image
            modified_input = imgs_tensor + modifier
            newimg = torch.sin(modified_input) * 0.5 + 0.5
            output = model(newimg)

            l2dist = torch.sum((newimg - imgs_tensor_original) ** 2, dim=1)

            real = output[:, 0]
            other = output[:, 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            loss = (CONST * loss1) + l2dist

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()

            with torch.no_grad():
                success_mask_inloop = compare(output)
                success_mask = torch.logical_or(success_mask_inloop, success_mask)
                o_success_mask  = torch.logical_or(o_success_mask, success_mask_inloop)

                update_mask = (l2dist < bestl2) & success_mask_inloop
                bestl2 = torch.where(update_mask, l2dist, bestl2)
                bestattack = torch.where(update_mask.unsqueeze(-1), newimg, bestattack)
                best_modifier = torch.where(update_mask.unsqueeze(-1), modifier.detach(), best_modifier)

                improved_mask = (loss1 < best_loss1)
                best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
                best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, best_unsuccessfull_attack)
                best_unsuccessfull_modifier = torch.where(improved_mask.unsqueeze(-1), modifier.detach(), best_unsuccessfull_modifier)

                update_mask2 = (l2dist < o_bestl2) & success_mask_inloop
                o_bestl2 = torch.where(update_mask2, l2dist, o_bestl2)
                o_bestattack = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack)

                improved_mask2 = (loss1 < o_best_loss1)
                o_best_loss1 = torch.where(improved_mask2, loss1, o_best_loss1)
                o_best_unsuccessfull_attack = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)


                if iteration == 0 or (iteration + 1) % 500 == 0:
                    current_lr = optimizer.param_groups[0]['lr']
                    current_momentum, current_momentum2 = optimizer.param_groups[0]['betas']
                    print(f"Iteration {iteration + 1:5}:\t Current LR = {current_lr:.4f} \t Current momentum = {current_momentum:.2f} \t Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t Loss = {total_loss.item():.4f} \t l2dist = {torch.mean(bestl2[success_mask]).item():.4f}")

                    print(f"Best results until now: \t success = {o_success_mask.sum().item()} \t l2dist = {torch.mean(o_bestl2[o_success_mask]).item():.4f}")

        failure_mask = ~success_mask
        upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
        lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

        upper_limit_mask = upper_bound < 1e9
        CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
        CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor



    print('')
    output = model(o_bestattack)
    final_success = compare(output)
    print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.4f}\t success Loss = {loss[final_success].sum().item():.4f} \t success l2dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
    print('')

    final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_output, combined_model, removal_array)
    print('')

In [ ]:
CW(combined_model, mal_x_batch, mal_y_batch, removal_array, device='cuda', initial_const = 1., binary_search_steps=8, cons_increase_factor=10., LR_function=lr_cosine_half,
                 max_lr=2.5, min_lr=1e-3, update_step=500, first_phase_iters=20000,second_phase_iters = 5000 , confidence=0.1, eps=1e-4)

## Rounded inloop - all const



In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim.lr_scheduler import LambdaLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def compare_round(mal_x_batch,newimg, model,removal_array):

    adv_rounded = final_rounding(mal_x_batch, newimg, model,removal_array)
    #l0dist = torch.sum((mal_x_batch - newimg)!=0, dim=1)
    l0dist = torch.sum(torch.abs(adv_rounded - mal_x_batch), dim=1)
    logits = model(adv_rounded)
    output = torch.argmax(logits, dim=1)

    return output == 0, l0dist

# Define the learning rate schedules (same as earlier)
def lr_linear(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start + t * (lr_end - lr_start) / T

def lr_exponential(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start * (lr_end / lr_start) ** (t / T)

def lr_cosine(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_end + (lr_start - lr_end) * 0.5 * (1 + torch.cos(torch.tensor(np.pi * t / T)))

def lr_average(t, T, lr_start=1e-3, lr_end=0.9):
    return 0.5 * (lr_linear(t, T, lr_start, lr_end) + lr_exponential(t, T, lr_start, lr_end))

def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))



def CW_l1(imgs, y, model, removal_array, device='cuda', initial_const = 1., binary_search_steps=8, cons_increase_factor=10., LR_function=lr_cosine_half,
                 max_lr=2., min_lr=1e-4,update_step=500, first_phase_iters=20000,second_phase_iters = 5000 , confidence=0.1, eps=1e-3):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - max_lr: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform.
    - confidence: Confidence parameter for adversarial loss computation.
    """

    def grad_hook(grad):
        return grad * non_fixed_features_mask

    # Preparing tensors
    imgs_tensor_original = imgs.clone().detach().to(dtype=torch.float32, device=device)
    imgs_tensor = torch.where(imgs.clone().detach().to(torch.uint8), torch.pi * 0.5, -torch.pi * 0.5).to(device)
    #labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)

    batch_size = imgs.shape[0]

    # Create masks for 0s and 1s
    zero_mask = (imgs == 0)  # Mask where values are 0
    one_mask = (imgs == 1)   # Mask where values are 1

    # Add eps to 0s and subtract eps from 1s
    imgs_tensor[zero_mask] += eps
    imgs_tensor[one_mask] -= eps

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))


    # Initialize tracking variables
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    # Total number of iterations
    total_iters = first_phase_iters + second_phase_iters

    for outer_step in range(binary_search_steps):
        print('------------------------------------------------------------------------------------------------------------------------------------')
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=min_lr)
        modifier.register_hook(grad_hook)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestattack = imgs.clone().detach()
        best_modifier = torch.zeros_like(modifier).detach()  # Best modifier
        best_unsuccessfull_modifier = torch.zeros_like(modifier).detach()

        best_loss1 = torch.full((batch_size,), float('inf'), device=device)
        best_unsuccessfull_attack = imgs.clone().detach()
        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(total_iters):

            # Phase 2: Fixed learning rate of 1e-3 (iterations 15,000 to 24,999)
            if iteration == first_phase_iters:
                print("************ second phase *************")
                final_success = (o_bestl2 != 1e10)
                final_output = torch.where(final_success.unsqueeze(-1), bestattack, best_unsuccessfull_attack)
                final_modifier = torch.where(final_success.unsqueeze(-1), best_modifier, best_unsuccessfull_modifier)

                modifier = torch.tensor(final_modifier.clone().detach(), requires_grad=True).to(device)
                optimizer = torch.optim.Adam([modifier], lr=1e-3)  # Use lr_min for 1000 iterations
                modifier.register_hook(grad_hook)

            if iteration < first_phase_iters:
                if iteration == 0 or (iteration + 1) % update_step == 0:
                    # Select your preferred learning rate schedule
                    current_lr = LR_function(iteration, first_phase_iters, min_lr, max_lr)  # You can switch to lr_linear, lr_exponential, or lr_cosine

                    # Update the optimizer's learning rate
                    for param_group in optimizer.param_groups:
                        param_group['lr'] = current_lr

            optimizer.zero_grad()

            # Attack Step: generate perturbed image
            modified_input = imgs_tensor + modifier
            newimg = torch.sin(modified_input) * 0.5 + 0.5
            output = model(newimg)

            l2dist = torch.sum((newimg - imgs_tensor_original) ** 2, dim=1)

            real = output[:, 0]
            other = output[:, 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            loss = (CONST * loss1) + l2dist

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()

            if (iteration + 1) % 25 == 0:
                with torch.no_grad():
                    success_mask_inloop, l0dist = compare_round(imgs_tensor_original,newimg, model,removal_array)
                    success_mask = torch.logical_or(success_mask_inloop, success_mask)
                    o_success_mask  = torch.logical_or(o_success_mask, success_mask_inloop)

                    update_mask = (l0dist < bestl2) & success_mask_inloop
                    bestl2 = torch.where(update_mask, l0dist, bestl2)
                    bestattack = torch.where(update_mask.unsqueeze(-1), newimg, bestattack)
                    best_modifier = torch.where(update_mask.unsqueeze(-1), modifier.detach(), best_modifier)

                    improved_mask = (loss1 < best_loss1)
                    best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
                    best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, best_unsuccessfull_attack)
                    best_unsuccessfull_modifier = torch.where(improved_mask.unsqueeze(-1), modifier.detach(), best_unsuccessfull_modifier)

                    update_mask2 = (l0dist < o_bestl2) & success_mask_inloop
                    o_bestl2 = torch.where(update_mask2, l0dist, o_bestl2)
                    o_bestattack = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack)

                    improved_mask2 = (loss1 < o_best_loss1)
                    o_best_loss1 = torch.where(improved_mask2, loss1, o_best_loss1)
                    o_best_unsuccessfull_attack = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)


                    if iteration == 0 or (iteration + 1) % 500 == 0:
                        current_lr = optimizer.param_groups[0]['lr']
                        current_momentum, current_momentum2 = optimizer.param_groups[0]['betas']
                        print(f"Iteration {iteration + 1:5}:\t Current LR = {current_lr:.4f} \t Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t Loss = {total_loss.item():.4f} \t l0dist = {torch.mean(bestl2[success_mask]).item():.4f}")

                        print(f"Best results until now: \t success = {o_success_mask.sum().item()} \t l0dist = {torch.mean(o_bestl2[o_success_mask]).item():.4f}")

        failure_mask = ~success_mask
        upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
        lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

        upper_limit_mask = upper_bound < 1e9
        CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
        CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor



    print('')
    final_success = (o_bestl2 != 1e10)
    print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.4f}\t success Loss = {loss[final_success].sum().item():.4f} \t success l0dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
    print('')

    final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_output, model, removal_array)
    print('')

    return final_output

In [ ]:
attack_params =  {'confidence':0.1, 'removal_array':removal_array,
                  'max_lr': 2.5, 'min_lr': 1e-3, 'initial_const': 1 , 'binary_search_steps' : 9,
                  'cons_increase_factor' : 10, 'first_phase_iters': 20000 , 'second_phase_iters' : 5000,
                  'LR_function' : lr_cosine_half, 'eps': 1e-4}
adv = adv_predict_rounding(test_loader, combined_model, CW_l1, device, **attack_params)

In [ ]:
# DIFFERENT MODEL : model_AT_rFGSM
attack_params =  {'confidence':0.1, 'removal_array':removal_array,
                  'max_lr': 2.5, 'min_lr': 1e-3, 'initial_const': 1 , 'binary_search_steps' : 8,
                  'cons_increase_factor' : 10, 'first_phase_iters': 20000 , 'second_phase_iters' : 5000,
                  'LR_function' : lr_cosine_half, 'eps': 1e-4}
adv = adv_predict_rounding(test_loader, model_AT_rFGSM, CW_l1, device, **attack_params)

## Step 5 — New sine + new distance function: the in-loop grid

# New sin function and new distance function:
newimg = torch.sin(modified_input) * 0.5 + 0.5

 Binary penalty to push values towards 0 or 1

        binary_penalty = torch.sum(newimg * (1 - newimg) , dim=1) # Element-wise penalty


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Generate x values
pertubation = np.linspace(-0.01, 1.01, 400)

# Define different alpha values between 0 and 1
alpha_values = np.linspace(0, 5, 11)


# Define the new function
def g(x, alpha):
    return -(x**2 + alpha * (x*(1-x))) #suppose imgs_tensor_original=0

# Define the derivative of the new function
def dg_dx(x, alpha):
    return -(2*x - alpha * (2*x - 1.))

# Plot the function and its derivative for different alpha values
plt.figure(figsize=(10, 20))

# Plot the original functions
plt.subplot(2, 1, 1)
for alpha in alpha_values:
    y = g(pertubation, alpha)
    plt.plot(pertubation, y, label=f'alpha={alpha:.2f}')
plt.title(r'Plot of $pertubation^2 - \alpha((pertubation-0.5)^2 - 0.25)$ for different $\alpha$')
plt.xlabel('pertubation')
plt.ylabel('g(pertubation)')
plt.axhline(0, color='black',linewidth=0.5)
plt.axvline(0, color='black',linewidth=0.5)
plt.legend()
plt.grid(True)

# Plot the derivatives
plt.subplot(2, 1, 2)
for alpha in alpha_values:
    y_derivative = dg_dx(pertubation, alpha)
    plt.plot(pertubation, y_derivative, label=f'alpha={alpha:.2f}')
plt.title(r"Plot of derivative of $pertubation^2 - \alpha((pertubation-0.5)^2 - 0.25)$")
plt.xlabel('pertubation')
plt.ylabel(r"$g'(pertubation)$")
plt.axhline(0, color='black',linewidth=0.5)
plt.axvline(0, color='black',linewidth=0.5)
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Since the environment was reset, we need to redefine the functions and variables
import numpy as np
import matplotlib.pyplot as plt

# Define the new trigonometric-based function
def h(x, alpha):
    return -(np.sin(x)**2 - alpha * ((np.sin(x) - 0.5)**2 - 0.25))

# Define the derivative of the new function
def dh_dx(x, alpha):
    return -2 * np.sin(x) * np.cos(x) + alpha * (2 * (np.sin(x) - 0.5) * np.cos(x))

# Generate x values
modified = np.linspace(0, np.pi/2., 400)

# Define different alpha values between 0 and 1
alpha_values = np.linspace(0, 5, 6)

# Plot the function and its derivative for different alpha values
plt.figure(figsize=(10, 20))

# Plot the original functions
plt.subplot(2, 1, 1)
for alpha in alpha_values:
    y = h(modified, alpha)
    plt.plot(modified, y, label=f'alpha={alpha:.2f}')
plt.title(r'Plot of $-(\sin^2(modified) - \alpha((\sin(modified)-0.5)^2 - 0.25))$ for different $\alpha$')
plt.xlabel('modified')
plt.ylabel('h(modified)')
plt.axhline(0, color='black',linewidth=0.5)
plt.axvline(0, color='black',linewidth=0.5)
plt.legend()
plt.grid(True)

# Plot the derivatives
plt.subplot(2, 1, 2)
for alpha in alpha_values:
    y_derivative = dh_dx(modified, alpha)
    plt.plot(modified, y_derivative, label=f'alpha={alpha:.2f}')
plt.title(r"Plot of derivative of $-(\sin^2(modified) - \alpha((\sin(modified)-0.5)^2 - 0.25))$")
plt.xlabel('modified')
plt.ylabel(r"$h'(modified)$")
plt.axhline(0, color='black',linewidth=0.5)
plt.axvline(0, color='black',linewidth=0.5)
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim.lr_scheduler import LambdaLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0


def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))



def CW_tanh_test(model, imgs, y, removal_array, device='cuda', const=100.0,
                 max_lr=2., min_lr=1e-4,update_step=1000, first_phase_iters=20000,second_phase_iters = 10000 , confidence=0.1, eps=1e-4, alpha=2.):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - max_lr: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform.
    - confidence: Confidence parameter for adversarial loss computation.
    """
    # Preparing tensors
    imgs_tensor_original = imgs.clone().detach().to(dtype=torch.float32, device=device)
    imgs_tensor = torch.where(imgs.clone().detach().to(torch.uint8), torch.pi * 0.5, -torch.pi * 0.5).to(device)
    labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)

    # Create masks for 0s and 1s
    zero_mask = (imgs == 0)  # Mask where values are 0
    one_mask = (imgs == 1)   # Mask where values are 1

    # Add eps to 0s and subtract eps from 1s
    imgs_tensor[zero_mask] += eps
    imgs_tensor[one_mask] -= eps

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
    previous_modifier = modifier.clone().detach()
    optimizer = torch.optim.Adam([modifier], lr=min_lr)


    def grad_hook(grad):
        return grad * non_fixed_features_mask

    modifier.register_hook(grad_hook)

    # Initialize tracking variables
    batch_size = imgs.shape[0]
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()
    o_best_modifier = torch.zeros_like(modifier).detach()  # Best modifier
    o_best_unsuccessfull_modifier = torch.zeros_like(modifier).detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    # Total number of iterations
    total_iters = first_phase_iters + second_phase_iters

    for iteration in range(total_iters):

        # Phase 2: Fixed learning rate of 1e-3 (iterations 15,000 to 24,999)
        if iteration == first_phase_iters:
            print('')
            output = model(o_bestattack)
            final_success = compare(output)
            print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.4f}\t success Loss = {loss[final_success].sum().item():.4f} \t success l2dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
            print('')
            final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
            final_modifier = torch.where(final_success.unsqueeze(-1), o_best_modifier, o_best_unsuccessfull_modifier)
            results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_output, combined_model, removal_array)
            print('')

            modifier = torch.tensor(final_modifier.clone().detach(), requires_grad=True).to(device)
            optimizer = torch.optim.Adam([modifier], lr=min_lr)  # Use lr_min for 1000 iterations
            modifier.register_hook(grad_hook)

        if iteration < first_phase_iters:

            if iteration == 0 or (iteration + 1) % update_step == 0:
                # Select your preferred learning rate schedule
                current_lr = lr_cosine_half(iteration, first_phase_iters, min_lr, max_lr)  # You can switch to lr_linear, lr_exponential, or lr_cosine

                # Update the optimizer's learning rate
                for param_group in optimizer.param_groups:
                    param_group['lr'] = current_lr


        optimizer.zero_grad()

        # Attack Step: generate perturbed image
        modified_input = imgs_tensor + modifier
        newimg = torch.sin(modified_input) * 0.5 + 0.5
        output = model(newimg)

        l2dist = torch.sum((newimg - imgs_tensor_original) ** 2 , dim=1)

        # Binary penalty to push values towards 0 or 1
        binary_penalty = torch.sum(newimg * (1 - newimg) , dim=1) # Element-wise penalty

        #pertubation = torch.abs((newimg - imgs_tensor_original))
        #l2dist = torch.sum(pertubation ** 2 - alpha * ((pertubation - 0.5)**2 - 0.25)  , dim=1)

        real = output[:, 0]
        other = output[:, 1]

        loss1 = torch.clamp((other - real + confidence), min=0.0)
        loss = (const * loss1) + l2dist
        loss = (CONST * loss1) + l2dist + alpha * binary_penalty
        total_loss = torch.sum(loss)
        total_loss.backward()
        optimizer.step()

        with torch.no_grad():
            success_mask_inloop = compare(output)
            success_mask = torch.logical_or(success_mask_inloop, success_mask)

            update_mask = (l2dist < o_bestl2) & success_mask_inloop
            o_bestl2 = torch.where(update_mask, l2dist, o_bestl2)
            o_bestattack = torch.where(update_mask.unsqueeze(-1), newimg, o_bestattack)
            o_best_modifier = torch.where(update_mask.unsqueeze(-1), modifier.detach(), o_best_modifier)

            improved_mask = (loss1 < o_best_loss1)
            o_best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
            o_best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)
            o_best_unsuccessfull_modifier = torch.where(improved_mask.unsqueeze(-1), modifier.detach(), o_best_unsuccessfull_modifier)

            if iteration == 0 or (iteration + 1) % 500 == 0:
                current_lr = optimizer.param_groups[0]['lr']
                current_momentum, current_momentum2 = optimizer.param_groups[0]['betas']
                print(f"Iteration {iteration + 1:5}:\t Current LR = {current_lr:.4f} \t Current momentum = {current_momentum:.2f} \t "
                      f"Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t "
                      f"Loss = {total_loss.item():.4f} \t l2dist = {torch.mean(o_bestl2[success_mask]).item():.4f}")

    print('')
    output = model(o_bestattack)
    final_success = compare(output)
    print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.4f}\t "
          f"success Loss = {loss[final_success].sum().item():.4f} \t success l2dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
    print('')

    final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_output, combined_model, removal_array)
    print('')

In [ ]:
CW_tanh_test(combined_model, mal_x_batch, mal_y_batch, removal_array, device=device, const=100.0,
                 max_lr=2.5, min_lr=1e-3, update_step=500, first_phase_iters=20000,second_phase_iters = 5000 , confidence=0.1, eps=1e-4, alpha=0)

## Rounded inloop - all const



In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim.lr_scheduler import LambdaLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def compare_round(mal_x_batch,newimg, model,removal_array):

    adv_rounded = final_rounding(mal_x_batch, newimg, model,removal_array)
    #l0dist = torch.sum((mal_x_batch - newimg)!=0, dim=1)
    l0dist = torch.sum(torch.abs(adv_rounded - mal_x_batch), dim=1)
    logits = model(adv_rounded)
    output = torch.argmax(logits, dim=1)

    return output == 0, l0dist

# Define the learning rate schedules (same as earlier)
def lr_linear(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start + t * (lr_end - lr_start) / T

def lr_exponential(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start * (lr_end / lr_start) ** (t / T)

def lr_cosine(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_end + (lr_start - lr_end) * 0.5 * (1 + torch.cos(torch.tensor(np.pi * t / T)))

def lr_average(t, T, lr_start=1e-3, lr_end=0.9):
    return 0.5 * (lr_linear(t, T, lr_start, lr_end) + lr_exponential(t, T, lr_start, lr_end))

def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))



def CW_l1(imgs, y, model, removal_array, device='cuda', initial_const = 1., binary_search_steps=8, cons_increase_factor=10., LR_function=lr_cosine_half,
                 max_lr=2., min_lr=1e-4,update_step=500, first_phase_iters=20000,second_phase_iters = 5000 , confidence=0.1, eps=1e-3, alpha=3.):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - max_lr: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform.
    - confidence: Confidence parameter for adversarial loss computation.
    """

    def grad_hook(grad):
        return grad * non_fixed_features_mask

    # Preparing tensors
    imgs_tensor_original = imgs.clone().detach().to(dtype=torch.float32, device=device)
    imgs_tensor = torch.where(imgs.clone().detach().to(torch.uint8), torch.pi * 0.5, -torch.pi * 0.5).to(device)
    #labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)

    batch_size = imgs.shape[0]

    # Create masks for 0s and 1s
    zero_mask = (imgs == 0)  # Mask where values are 0
    one_mask = (imgs == 1)   # Mask where values are 1

    # Add eps to 0s and subtract eps from 1s
    imgs_tensor[zero_mask] += eps
    imgs_tensor[one_mask] -= eps

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))


    # Initialize tracking variables
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    # Total number of iterations
    total_iters = first_phase_iters + second_phase_iters

    for outer_step in range(binary_search_steps):
        print('------------------------------------------------------------------------------------------------------------------------------------')
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=min_lr)
        modifier.register_hook(grad_hook)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestattack = imgs.clone().detach()
        best_modifier = torch.zeros_like(modifier).detach()  # Best modifier
        best_unsuccessfull_modifier = torch.zeros_like(modifier).detach()

        best_loss1 = torch.full((batch_size,), float('inf'), device=device)
        best_unsuccessfull_attack = imgs.clone().detach()
        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(total_iters):

            # Phase 2: Fixed learning rate of 1e-3 (iterations 15,000 to 24,999)
            if iteration == first_phase_iters:
                print("************ second phase *************")
                final_success = (o_bestl2 != 1e10)
                final_output = torch.where(final_success.unsqueeze(-1), bestattack, best_unsuccessfull_attack)
                final_modifier = torch.where(final_success.unsqueeze(-1), best_modifier, best_unsuccessfull_modifier)

                modifier = torch.tensor(final_modifier.clone().detach(), requires_grad=True).to(device)
                optimizer = torch.optim.Adam([modifier], lr=1e-3)  # Use lr_min for 1000 iterations
                modifier.register_hook(grad_hook)

            if iteration < first_phase_iters:
                if iteration == 0 or (iteration + 1) % update_step == 0:
                    # Select your preferred learning rate schedule
                    current_lr = LR_function(iteration, first_phase_iters, min_lr, max_lr)  # You can switch to lr_linear, lr_exponential, or lr_cosine

                    # Update the optimizer's learning rate
                    for param_group in optimizer.param_groups:
                        param_group['lr'] = current_lr

            optimizer.zero_grad()

            # Attack Step: generate perturbed image
            modified_input = imgs_tensor + modifier
            newimg = torch.sin(modified_input) * 0.5 + 0.5
            output = model(newimg)

            #pertubation = torch.abs((newimg - imgs_tensor_original))
            #l2dist = torch.sum(pertubation ** 2 - alpha * ((pertubation - 0.5)**2 - 0.25)  , dim=1)

            l2dist = torch.sum((newimg - imgs_tensor_original) ** 2 , dim=1)

            # Binary penalty to push values towards 0 or 1
            binary_penalty = torch.sum(newimg * (1 - newimg) , dim=1) # Element-wise penalty

            real = output[:, 0]
            other = output[:, 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            loss = (CONST * loss1) + l2dist + alpha * binary_penalty

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()

            if (iteration + 1) % 25 == 0:
                with torch.no_grad():
                    success_mask_inloop, l0dist = compare_round(imgs_tensor_original,newimg, model,removal_array)
                    success_mask = torch.logical_or(success_mask_inloop, success_mask)
                    o_success_mask  = torch.logical_or(o_success_mask, success_mask_inloop)

                    update_mask = (l0dist < bestl2) & success_mask_inloop
                    bestl2 = torch.where(update_mask, l0dist, bestl2)
                    bestattack = torch.where(update_mask.unsqueeze(-1), newimg, bestattack)
                    best_modifier = torch.where(update_mask.unsqueeze(-1), modifier.detach(), best_modifier)

                    improved_mask = (loss1 < best_loss1)
                    best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
                    best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, best_unsuccessfull_attack)
                    best_unsuccessfull_modifier = torch.where(improved_mask.unsqueeze(-1), modifier.detach(), best_unsuccessfull_modifier)

                    update_mask2 = (l0dist < o_bestl2) & success_mask_inloop
                    o_bestl2 = torch.where(update_mask2, l0dist, o_bestl2)
                    o_bestattack = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack)

                    improved_mask2 = (loss1 < o_best_loss1)
                    o_best_loss1 = torch.where(improved_mask2, loss1, o_best_loss1)
                    o_best_unsuccessfull_attack = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)


                    if iteration == 0 or (iteration + 1) % 1000 == 0:
                        current_lr = optimizer.param_groups[0]['lr']
                        current_momentum, current_momentum2 = optimizer.param_groups[0]['betas']
                        print(f"Iteration {iteration + 1:5}:\t Current LR = {current_lr:.4f} \t Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t Loss = {total_loss.item():.4f} \t l0dist = {torch.mean(bestl2[success_mask]).item():.4f}")

                        print(f"Best results until now: \t success = {o_success_mask.sum().item()} \t l0dist = {torch.mean(o_bestl2[o_success_mask]).item():.4f}")

        failure_mask = ~success_mask
        upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
        lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

        upper_limit_mask = upper_bound < 1e9
        CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
        CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor



    print('')
    final_success = (o_bestl2 != 1e10)
    print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.4f}\t success Loss = {loss[final_success].sum().item():.4f} \t success l0dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
    print('')

    final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_output, model, removal_array)
    print('')

    return final_output

In [ ]:
import itertools

# Define hyperparameter grid
param_grid = {
    'alpha': [0.,0.5,1.,1.5,2.,2.5,3.],
    'eps': [0,1e-4],
}

for params in itertools.product(*param_grid.values()):
    # Check if this set of parameters gives the best result
    print('**********************************************************************************************************************************************')
    alpha, eps = params
    print(f"Params:\t alpha={alpha}\t eps={eps}")
    print('-------------------------------')
    attack_params =  {'confidence':0.1, 'removal_array':removal_array,
                  'max_lr': 2.5, 'min_lr': 1e-3, 'initial_const': 100 , 'binary_search_steps' : 1,
                  'cons_increase_factor' : 10, 'first_phase_iters': 20000 , 'second_phase_iters' : 5000,
                  'LR_function' : lr_cosine_half, 'eps': eps, 'alpha': alpha}
    adv = adv_predict_rounding(test_loader, combined_model, CW_l1, device, **attack_params)

In [ ]:
attack_params =  {'confidence':0.1, 'removal_array':removal_array,
                  'max_lr': 2.5, 'min_lr': 1e-3, 'initial_const': 1e9 , 'binary_search_steps' : 1,
                  'cons_increase_factor' : 10, 'first_phase_iters': 20000 , 'second_phase_iters' : 5000,
                  'LR_function' : lr_cosine_half, 'eps': 1e-4, 'alpha': 0}
adv = adv_predict_rounding(test_loader, combined_model, CW_l1, device, **attack_params)

In [ ]:
attack_params =  {'confidence':0.2, 'removal_array':removal_array,
                  'max_lr': 2.5, 'min_lr': 1e-3, 'initial_const': 1e9 , 'binary_search_steps' : 1,
                  'cons_increase_factor' : 10, 'first_phase_iters': 20000 , 'second_phase_iters' : 5000,
                  'LR_function' : lr_cosine_half, 'eps': 1e-4, 'alpha': 0}
adv = adv_predict_rounding(test_loader, combined_model, CW_l1, device, **attack_params)

## Step 6 — Refining the function, and the tanh comparison

# New sin function and new distance function:
newimg = torch.sin(modified_input) * 0.5 + 0.5

 Binary penalty to push values towards 0 or 1

        binary_penalty = torch.minimum(x, 1 - x) # Element-wise penalty
        OR
        binary_penalty = (x - torch.round(x)) ** p


In [ ]:
# Adding the new function and its derivative and plotting it

# Define the new function g2 with np.minimum
def g2_min(x, alpha):
    """Computes the function g2 using np.minimum."""
    return -(x**2 + alpha * np.minimum(x, 1 - x))

# Compute the derivative of g2_min with respect to x
def g2_min_derivative(x, alpha):
    """Computes the derivative of g2 with np.minimum."""
    # The derivative of np.minimum(x, 1-x) is piecewise: 1 for x < 0.5, -1 for x > 0.5
    min_derivative = np.where(x < 0.5, 1, -1)
    return -(2 * x + alpha * min_derivative)

# Plot the new function and its derivative
def plot_new_function_and_derivative(perturbation, alpha_values):
    """Plots the function g2 (with np.minimum) and its derivative for different alpha values."""
    plt.figure(figsize=(10, 12))

    # Plot g2_min for different alpha values
    plt.subplot(2, 1, 1)
    for alpha in alpha_values:
        y = g2_min(perturbation, alpha)
        plt.plot(perturbation, y, label=f'alpha={alpha:.2f}')
    plt.title(r"Function $g_{min}(perturbation)$ for different $\alpha$ values")
    plt.xlabel('perturbation')
    plt.ylabel('$g_{min}(perturbation)$')
    plt.axhline(0, color='black', linewidth=0.5)
    plt.axvline(0, color='black', linewidth=0.5)
    plt.legend()
    plt.grid(True)

    # Plot the derivative of g2_min for different alpha values
    plt.subplot(2, 1, 2)
    for alpha in alpha_values:
        dy = g2_min_derivative(perturbation, alpha)
        plt.plot(perturbation, dy, label=f'alpha={alpha:.2f}')
    plt.title(r"Derivative $g'_{min}(perturbation)$ for different $\alpha$ values")
    plt.xlabel('perturbation')
    plt.ylabel(r"$g'_{min}(perturbation)$")
    plt.axhline(0, color='black', linewidth=0.5)
    plt.axvline(0, color='black', linewidth=0.5)
    plt.legend()
    plt.grid(True)

    # Adjust layout for better visualization
    plt.tight_layout()
    plt.show()

# Call the new plotting function for g2_min and its derivative
plot_new_function_and_derivative(perturbation, alpha_values)

In [ ]:
# Improved version of the code for better structure, clarity, and visualization

import numpy as np
import matplotlib.pyplot as plt

# Generate x values
perturbation = np.linspace(-0.01, 1.01, 400)

# Define different alpha values between 0 and 3
alpha_values = np.linspace(0, 3, 7)

# Define a function for the modified Lp-norm (Pseudo-Hamming Distance)
def pseudo_hamming_distance(p, x):
    """Computes the pseudo-Hamming distance with norm p."""
    return (x - np.round(x)) ** p

# Function g2 that incorporates quadratic and pseudo-Hamming distance
def g2(x, alpha):
    """Computes the function g2 with perturbation x and alpha."""
    return -(x**2 + alpha * pseudo_hamming_distance(2, x))

# Compute the derivative of g2 with respect to x
def g2_derivative(x, alpha):
    """Computes the derivative of g2."""
    # Derivative of the pseudo-Hamming distance
    derivative_hamming = 2 * (x - np.round(x))
    return -(2 * x + alpha * derivative_hamming)

# Improved plotting function
def plot_function_and_derivative(perturbation, alpha_values):
    """Plots the function g2 and its derivative for different alpha values."""
    plt.figure(figsize=(10, 12))

    # Plot g2 for different alpha values
    plt.subplot(2, 1, 1)
    for alpha in alpha_values:
        y = g2(perturbation, alpha)
        plt.plot(perturbation, y, label=f'alpha={alpha:.2f}')
    plt.title(r"Function $g(perturbation)$ for different $\alpha$ values")
    plt.xlabel('perturbation')
    plt.ylabel('g(perturbation)')
    plt.axhline(0, color='black', linewidth=0.5)
    plt.axvline(0, color='black', linewidth=0.5)
    plt.legend()
    plt.grid(True)

    # Plot the derivative of g2 for different alpha values
    plt.subplot(2, 1, 2)
    for alpha in alpha_values:
        dy = g2_derivative(perturbation, alpha)
        plt.plot(perturbation, dy, label=f'alpha={alpha:.2f}')
    plt.title(r"Derivative $g'(perturbation)$ for different $\alpha$ values")
    plt.xlabel('perturbation')
    plt.ylabel(r"$g'(perturbation)$")
    plt.axhline(0, color='black', linewidth=0.5)
    plt.axvline(0, color='black', linewidth=0.5)
    plt.legend()
    plt.grid(True)

    # Adjust layout for better visualization
    plt.tight_layout()
    plt.show()

# Call the plotting function
plot_function_and_derivative(perturbation, alpha_values)

In [ ]:
# Since the environment was reset, we need to redefine the functions and variables
import numpy as np
import matplotlib.pyplot as plt

# Define the new trigonometric-based function
def h(x, alpha):
    return -(np.sin(x)**2 + alpha * ((np.sin(x) - np.round(np.sin(x)))**2 ))

# Define the derivative of the new function
def dh_dx(x, alpha):
    return -(2 * np.sin(x) * np.cos(x) + alpha * (2 * (np.sin(x) - np.round(np.sin(x))) * np.cos(x)))

# Generate x values
modified = np.linspace(0, np.pi/2., 400)

# Define different alpha values between 0 and 1
alpha_values = np.linspace(0, 3, 7)

# Plot the function and its derivative for different alpha values
plt.figure(figsize=(10, 20))

# Plot the original functions
plt.subplot(2, 1, 1)
for alpha in alpha_values:
    y = h(modified, alpha)
    plt.plot(modified, y, label=f'alpha={alpha:.2f}')
plt.xlabel('modified')
plt.ylabel('h(modified)')
plt.axhline(0, color='black',linewidth=0.5)
plt.axvline(0, color='black',linewidth=0.5)
plt.legend()
plt.grid(True)

# Plot the derivatives
plt.subplot(2, 1, 2)
for alpha in alpha_values:
    y_derivative = dh_dx(modified, alpha)
    plt.plot(modified, y_derivative, label=f'alpha={alpha:.2f}')

plt.xlabel('modified')
plt.ylabel(r"$h'(modified)$")
plt.axhline(0, color='black',linewidth=0.5)
plt.axvline(0, color='black',linewidth=0.5)
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Rounded inloop - all const



In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim.lr_scheduler import LambdaLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def compare_round(mal_x_batch,newimg, model,removal_array):

    adv_rounded = final_rounding(mal_x_batch, newimg, model,removal_array)
    #l0dist = torch.sum((mal_x_batch - newimg)!=0, dim=1)
    l0dist = torch.sum(torch.abs(adv_rounded - mal_x_batch), dim=1)
    logits = model(adv_rounded)
    output = torch.argmax(logits, dim=1)

    return output == 0, l0dist

# Define the learning rate schedules (same as earlier)
def lr_linear(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start + t * (lr_end - lr_start) / T

def lr_exponential(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start * (lr_end / lr_start) ** (t / T)

def lr_cosine(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_end + (lr_start - lr_end) * 0.5 * (1 + torch.cos(torch.tensor(np.pi * t / T)))

def lr_average(t, T, lr_start=1e-3, lr_end=0.9):
    return 0.5 * (lr_linear(t, T, lr_start, lr_end) + lr_exponential(t, T, lr_start, lr_end))

def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))



def CW_l1(imgs, y, model, removal_array, device='cuda', initial_const = 1., binary_search_steps=8, cons_increase_factor=10., LR_function=lr_cosine_half,
                 max_lr=2., min_lr=1e-4,update_step=500, first_phase_iters=20000,second_phase_iters = 5000 , confidence=0.1, eps=1e-3, alpha=3.):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - max_lr: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform.
    - confidence: Confidence parameter for adversarial loss computation.
    """

    def grad_hook(grad):
        return grad * non_fixed_features_mask

    # Preparing tensors
    imgs_tensor_original = imgs.clone().detach().to(dtype=torch.float32, device=device)
    imgs_tensor = torch.where(imgs.clone().detach().to(torch.uint8), torch.pi * 0.5, -torch.pi * 0.5).to(device)
    #labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)

    batch_size = imgs.shape[0]

    # Create masks for 0s and 1s
    zero_mask = (imgs == 0)  # Mask where values are 0
    one_mask = (imgs == 1)   # Mask where values are 1

    # Add eps to 0s and subtract eps from 1s
    imgs_tensor[zero_mask] += eps
    imgs_tensor[one_mask] -= eps

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))


    # Initialize tracking variables
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    # Total number of iterations
    total_iters = first_phase_iters + second_phase_iters

    for outer_step in range(binary_search_steps):
        print('------------------------------------------------------------------------------------------------------------------------------------')
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=min_lr)
        modifier.register_hook(grad_hook)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestattack = imgs.clone().detach()
        best_modifier = torch.zeros_like(modifier).detach()  # Best modifier
        best_unsuccessfull_modifier = torch.zeros_like(modifier).detach()

        best_loss1 = torch.full((batch_size,), float('inf'), device=device)
        best_unsuccessfull_attack = imgs.clone().detach()
        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(total_iters):

            # Phase 2: Fixed learning rate of 1e-3 (iterations 15,000 to 24,999)
            if iteration == first_phase_iters:
                print("************ second phase *************")
                final_success = (o_bestl2 != 1e10)
                final_output = torch.where(final_success.unsqueeze(-1), bestattack, best_unsuccessfull_attack)
                final_modifier = torch.where(final_success.unsqueeze(-1), best_modifier, best_unsuccessfull_modifier)

                modifier = torch.tensor(final_modifier.clone().detach(), requires_grad=True).to(device)
                optimizer = torch.optim.Adam([modifier], lr=1e-3)  # Use lr_min for 1000 iterations
                modifier.register_hook(grad_hook)

            if iteration < first_phase_iters:
                if iteration == 0 or (iteration + 1) % update_step == 0:
                    # Select your preferred learning rate schedule
                    current_lr = LR_function(iteration, first_phase_iters, min_lr, max_lr)  # You can switch to lr_linear, lr_exponential, or lr_cosine

                    # Update the optimizer's learning rate
                    for param_group in optimizer.param_groups:
                        param_group['lr'] = current_lr

            optimizer.zero_grad()

            # Attack Step: generate perturbed image
            modified_input = imgs_tensor + modifier
            newimg = torch.sin(modified_input) * 0.5 + 0.5
            output = model(newimg)

            #pertubation = torch.abs((newimg - imgs_tensor_original))
            #l2dist = torch.sum(pertubation ** 2 - alpha * ((pertubation - 0.5)**2 - 0.25)  , dim=1)

            l2dist = torch.sum((newimg - imgs_tensor_original) ** 2 , dim=1)

            # Binary penalty to push values towards 0 or 1
            binary_penalty = torch.sum((newimg - torch.round(newimg)) ** 2 , dim=1) # Element-wise penalty

            real = output[:, 0]
            other = output[:, 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            loss = (CONST * loss1) + l2dist + alpha * binary_penalty

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()

            if (iteration + 1) % 25 == 0:
                with torch.no_grad():
                    success_mask_inloop, l0dist = compare_round(imgs_tensor_original,newimg, model,removal_array)
                    success_mask = torch.logical_or(success_mask_inloop, success_mask)
                    o_success_mask  = torch.logical_or(o_success_mask, success_mask_inloop)

                    update_mask = (l0dist < bestl2) & success_mask_inloop
                    bestl2 = torch.where(update_mask, l0dist, bestl2)
                    bestattack = torch.where(update_mask.unsqueeze(-1), newimg, bestattack)
                    best_modifier = torch.where(update_mask.unsqueeze(-1), modifier.detach(), best_modifier)

                    improved_mask = (loss1 < best_loss1)
                    best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
                    best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, best_unsuccessfull_attack)
                    best_unsuccessfull_modifier = torch.where(improved_mask.unsqueeze(-1), modifier.detach(), best_unsuccessfull_modifier)

                    update_mask2 = (l0dist < o_bestl2) & success_mask_inloop
                    o_bestl2 = torch.where(update_mask2, l0dist, o_bestl2)
                    o_bestattack = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack)

                    improved_mask2 = (loss1 < o_best_loss1)
                    o_best_loss1 = torch.where(improved_mask2, loss1, o_best_loss1)
                    o_best_unsuccessfull_attack = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)


                    if iteration == 0 or (iteration + 1) % 1000 == 0:
                        current_lr = optimizer.param_groups[0]['lr']
                        current_momentum, current_momentum2 = optimizer.param_groups[0]['betas']
                        print(f"Iteration {iteration + 1:5}:\t Current LR = {current_lr:.4f} \t Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t Loss = {total_loss.item():.4f} \t l0dist = {torch.mean(bestl2[success_mask]).item():.4f}")

                        #print(f"Best results until now: \t success = {o_success_mask.sum().item()} \t l0dist = {torch.mean(o_bestl2[o_success_mask]).item():.4f}")

        failure_mask = ~success_mask
        upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
        lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

        upper_limit_mask = upper_bound < 1e9
        CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
        CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor



    print('')
    final_success = (o_bestl2 != 1e10)
    print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.4f}\t success Loss = {loss[final_success].sum().item():.4f} \t success l0dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
    print('')

    final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_output, model, removal_array)
    print('')

    return final_output

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim.lr_scheduler import LambdaLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def compare_round(mal_x_batch,newimg, model,removal_array):

    adv_rounded = final_rounding(mal_x_batch, newimg, model,removal_array)
    #l0dist = torch.sum((mal_x_batch - newimg)!=0, dim=1)
    l0dist = torch.sum(torch.abs(adv_rounded - mal_x_batch), dim=1)
    logits = model(adv_rounded)
    output = torch.argmax(logits, dim=1)

    return output == 0, l0dist

# Define the learning rate schedules (same as earlier)
def lr_linear(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start + t * (lr_end - lr_start) / T

def lr_exponential(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start * (lr_end / lr_start) ** (t / T)

def lr_cosine(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_end + (lr_start - lr_end) * 0.5 * (1 + torch.cos(torch.tensor(np.pi * t / T)))

def lr_average(t, T, lr_start=1e-3, lr_end=0.9):
    return 0.5 * (lr_linear(t, T, lr_start, lr_end) + lr_exponential(t, T, lr_start, lr_end))

def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))



def CW_l1(imgs, y, model, removal_array, device='cuda', initial_const = 1., binary_search_steps=8, cons_increase_factor=10., LR_function=lr_cosine_half,
                 max_lr=2., min_lr=1e-4,update_step=500, first_phase_iters=20000,second_phase_iters = 5000 , confidence=0.1, eps=1e-3, alpha=3.):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - max_lr: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform.
    - confidence: Confidence parameter for adversarial loss computation.
    """

    def grad_hook(grad):
        return grad * non_fixed_features_mask

    # Preparing tensors
    imgs_tensor_original = imgs.clone().detach().to(dtype=torch.float32, device=device)
    imgs_tensor = torch.where(imgs.clone().detach().to(torch.uint8), torch.pi * 0.5, -torch.pi * 0.5).to(device)
    #labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)

    batch_size = imgs.shape[0]

    # Create masks for 0s and 1s
    zero_mask = (imgs == 0)  # Mask where values are 0
    one_mask = (imgs == 1)   # Mask where values are 1

    # Add eps to 0s and subtract eps from 1s
    imgs_tensor[zero_mask] += eps
    imgs_tensor[one_mask] -= eps

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))


    # Initialize tracking variables
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    # Total number of iterations
    total_iters = first_phase_iters + second_phase_iters

    for outer_step in range(binary_search_steps):
        print('------------------------------------------------------------------------------------------------------------------------------------')
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=min_lr)
        modifier.register_hook(grad_hook)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestattack = imgs.clone().detach()
        best_modifier = torch.zeros_like(modifier).detach()  # Best modifier
        best_unsuccessfull_modifier = torch.zeros_like(modifier).detach()

        best_loss1 = torch.full((batch_size,), float('inf'), device=device)
        best_unsuccessfull_attack = imgs.clone().detach()
        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(total_iters):

            # Phase 2: Fixed learning rate of 1e-3 (iterations 15,000 to 24,999)
            if iteration == first_phase_iters:
                print("************ second phase *************")
                final_success = (o_bestl2 != 1e10)
                final_output = torch.where(final_success.unsqueeze(-1), bestattack, best_unsuccessfull_attack)
                final_modifier = torch.where(final_success.unsqueeze(-1), best_modifier, best_unsuccessfull_modifier)

                modifier = torch.tensor(final_modifier.clone().detach(), requires_grad=True).to(device)
                optimizer = torch.optim.Adam([modifier], lr=1e-3)  # Use lr_min for 1000 iterations
                modifier.register_hook(grad_hook)

            if iteration < first_phase_iters:
                if iteration == 0 or (iteration + 1) % update_step == 0:
                    # Select your preferred learning rate schedule
                    current_lr = LR_function(iteration, first_phase_iters, min_lr, max_lr)  # You can switch to lr_linear, lr_exponential, or lr_cosine

                    # Update the optimizer's learning rate
                    for param_group in optimizer.param_groups:
                        param_group['lr'] = current_lr

            optimizer.zero_grad()

            # Attack Step: generate perturbed image
            modified_input = imgs_tensor + modifier
            newimg = torch.sin(modified_input) * 0.5 + 0.5
            output = model(newimg)

            #pertubation = torch.abs((newimg - imgs_tensor_original))
            #l2dist = torch.sum(pertubation ** 2 - alpha * ((pertubation - 0.5)**2 - 0.25)  , dim=1)

            l2dist = torch.sum((newimg - imgs_tensor_original) ** 2 , dim=1)

            # Binary penalty to push values towards 0 or 1
            binary_penalty = torch.sum(torch.minimum(newimg, 1-newimg), dim=1) # Element-wise penalty

            real = output[:, 0]
            other = output[:, 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            loss = (CONST * loss1) + l2dist + alpha * binary_penalty

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()

            if (iteration + 1) % 25 == 0:
                with torch.no_grad():
                    success_mask_inloop, l0dist = compare_round(imgs_tensor_original,newimg, model,removal_array)
                    success_mask = torch.logical_or(success_mask_inloop, success_mask)
                    o_success_mask  = torch.logical_or(o_success_mask, success_mask_inloop)

                    update_mask = (l0dist < bestl2) & success_mask_inloop
                    bestl2 = torch.where(update_mask, l0dist, bestl2)
                    bestattack = torch.where(update_mask.unsqueeze(-1), newimg, bestattack)
                    best_modifier = torch.where(update_mask.unsqueeze(-1), modifier.detach(), best_modifier)

                    improved_mask = (loss1 < best_loss1)
                    best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
                    best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, best_unsuccessfull_attack)
                    best_unsuccessfull_modifier = torch.where(improved_mask.unsqueeze(-1), modifier.detach(), best_unsuccessfull_modifier)

                    update_mask2 = (l0dist < o_bestl2) & success_mask_inloop
                    o_bestl2 = torch.where(update_mask2, l0dist, o_bestl2)
                    o_bestattack = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack)

                    improved_mask2 = (loss1 < o_best_loss1)
                    o_best_loss1 = torch.where(improved_mask2, loss1, o_best_loss1)
                    o_best_unsuccessfull_attack = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)


                    if iteration == 0 or (iteration + 1) % 1000 == 0:
                        current_lr = optimizer.param_groups[0]['lr']
                        current_momentum, current_momentum2 = optimizer.param_groups[0]['betas']
                        print(f"Iteration {iteration + 1:5}:\t Current LR = {current_lr:.4f} \t Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t Loss = {total_loss.item():.4f} \t l0dist = {torch.mean(bestl2[success_mask]).item():.4f}")

                        #print(f"Best results until now: \t success = {o_success_mask.sum().item()} \t l0dist = {torch.mean(o_bestl2[o_success_mask]).item():.4f}")

        failure_mask = ~success_mask
        upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
        lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

        upper_limit_mask = upper_bound < 1e9
        CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
        CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor



    print('')
    final_success = (o_bestl2 != 1e10)
    print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.4f}\t success Loss = {loss[final_success].sum().item():.4f} \t success l0dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
    print('')

    final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_output, model, removal_array)
    print('')

    return final_output

In [ ]:
import itertools

# Define hyperparameter grid
param_grid = {
    'alpha': [0.5,1.,1.5,2.,2.5,3.],
    'eps': [0,1e-4],
}

for params in itertools.product(*param_grid.values()):
    # Check if this set of parameters gives the best result
    print('**********************************************************************************************************************************************')
    alpha, eps = params
    print(f"Params:\t alpha={alpha}\t eps={eps}")
    print('-------------------------------')
    attack_params =  {'confidence':0.1, 'removal_array':removal_array,
                  'max_lr': 2.5, 'min_lr': 1e-3, 'initial_const': 100 , 'binary_search_steps' : 1,
                  'cons_increase_factor' : 10, 'first_phase_iters': 20000 , 'second_phase_iters' : 5000,
                  'LR_function' : lr_cosine_half, 'eps': eps, 'alpha': alpha}
    adv = adv_predict_rounding(test_loader, combined_model, CW_l1, device, **attack_params)

# tanh

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim.lr_scheduler import LambdaLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.
    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def torch_arctanh(x, eps=1e-6):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

# Define the learning rate schedules (same as earlier)
def lr_linear(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start + t * (lr_end - lr_start) / T

def lr_exponential(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start * (lr_end / lr_start) ** (t / T)

def lr_cosine(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_end + (lr_start - lr_end) * 0.5 * (1 + torch.cos(torch.tensor(np.pi * t / T)))

def lr_average(t, T, lr_start=1e-3, lr_end=0.9):
    return 0.5 * (lr_linear(t, T, lr_start, lr_end) + lr_exponential(t, T, lr_start, lr_end))

def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))



def CW_tanh_test(model, imgs, y, removal_array, device='cuda', const=100.0, LR_function=lr_exponential,
                 max_lr=2., min_lr=1e-4,update_step=1000, first_phase_iters=20000,second_phase_iters = 10000 , confidence=0.1, eps=1e-3):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - max_lr: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform.
    - confidence: Confidence parameter for adversarial loss computation.
    """
    # Preparing tensors
    imgs_tensor = torch_arctanh(imgs, eps).clone().detach().to(dtype=torch.float32, device=device)
    imgs_tensor_original = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
    optimizer = torch.optim.Adam([modifier], lr=min_lr)


    def grad_hook(grad):
        return grad * non_fixed_features_mask

    modifier.register_hook(grad_hook)

    # Initialize tracking variables
    batch_size = imgs.shape[0]
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()
    o_best_modifier = torch.zeros_like(modifier).detach()  # Best modifier
    o_best_unsuccessfull_modifier = torch.zeros_like(modifier).detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    # Total number of iterations
    total_iters = first_phase_iters + second_phase_iters

    for iteration in range(total_iters):

        # Phase 2: Fixed learning rate of 1e-3 (iterations 15,000 to 24,999)
        if iteration == first_phase_iters:
            print('')
            output = model(o_bestattack)
            final_success = compare(output)
            print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.4f}\t success Loss = {loss[final_success].sum().item():.4f} \t success l2dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
            print('')
            final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
            final_modifier = torch.where(final_success.unsqueeze(-1), o_best_modifier, o_best_unsuccessfull_modifier)
            results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_output, combined_model, removal_array)
            print('')

            modifier = torch.tensor(final_modifier.clone().detach(), requires_grad=True).to(device)
            optimizer = torch.optim.Adam([modifier], lr=1e-3)  # Use lr_min for 1000 iterations
            modifier.register_hook(grad_hook)

        if iteration < first_phase_iters:

            if iteration == 0 or (iteration + 1) % update_step == 0:
                # Select your preferred learning rate schedule
                current_lr = LR_function(iteration, first_phase_iters, min_lr, max_lr)  # You can switch to lr_linear, lr_exponential, or lr_cosine

                # Update the optimizer's learning rate
                for param_group in optimizer.param_groups:
                    param_group['lr'] = current_lr


        optimizer.zero_grad()

        # Attack Step: generate perturbed image
        modified_input = imgs_tensor + modifier
        newimg = torch.tanh(modified_input) * 0.5 + 0.5
        output = model(newimg)

        l2dist = torch.sum((newimg - imgs_tensor_original) ** 2, dim=1)

        real = output[:, 0]
        other = output[:, 1]

        loss1 = torch.clamp((other - real + confidence), min=0.0)
        loss = (const * loss1) + l2dist

        total_loss = torch.sum(loss)
        total_loss.backward()
        optimizer.step()


        with torch.no_grad():
            success_mask_inloop = compare(output)
            success_mask = torch.logical_or(success_mask_inloop, success_mask)

            update_mask = (l2dist < o_bestl2) & success_mask_inloop
            o_bestl2 = torch.where(update_mask, l2dist, o_bestl2)
            o_bestattack = torch.where(update_mask.unsqueeze(-1), newimg, o_bestattack)
            o_best_modifier = torch.where(update_mask.unsqueeze(-1), modifier.detach(), o_best_modifier)

            improved_mask = (loss1 < o_best_loss1)
            o_best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
            o_best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)
            o_best_unsuccessfull_modifier = torch.where(improved_mask.unsqueeze(-1), modifier.detach(), o_best_unsuccessfull_modifier)

            if iteration == 0 or (iteration + 1) % 500 == 0:
                current_lr = optimizer.param_groups[0]['lr']
                current_momentum, current_momentum2 = optimizer.param_groups[0]['betas']
                print(f"Iteration {iteration + 1:5}:\t Current LR = {current_lr:.4f} \t Current momentum = {current_momentum:.2f} \t "
                      f"Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t "
                      f"Loss = {total_loss.item():.4f} \t l2dist = {torch.mean(o_bestl2[success_mask]).item():.4f}")

    print('')
    output = model(o_bestattack)
    final_success = compare(output)
    print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.4f}\t "
          f"success Loss = {loss[final_success].sum().item():.4f} \t success l2dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
    print('')

    final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_output, combined_model, removal_array)
    print('')

In [ ]:
import itertools

# Define hyperparameter grid
param_grid = {
    'max_lr': [1.,1.4,1.8,2.2],
    'LR_function': [lr_cosine_half],
}


for params in itertools.product(*param_grid.values()):
    # Check if this set of parameters gives the best result
    print('**********************************************************************************************************************************************')
    max_lr, LR_function = params
    print(f"Params:\t max_lr={max_lr}\t LR_function={LR_function.__name__}")
    print('-------------------------------')
    CW_tanh_test(combined_model, mal_x_batch, mal_y_batch, removal_array, device='cuda', const=100.0, LR_function=LR_function,
                 max_lr=max_lr, min_lr=1e-3, update_step=500, first_phase_iters=20000,second_phase_iters = 5000 , confidence=0.1, eps=1e-3)

Fixed LR

In [ ]:
import itertools

# Define hyperparameter grid
param_grid = {
    'max_lr': [0.05,0.1,0.5,1.,1.5,2.],
    'LR_function': [lr_cosine_half],
}


for params in itertools.product(*param_grid.values()):
    # Check if this set of parameters gives the best result
    print('**********************************************************************************************************************************************')
    max_lr, LR_function = params
    print(f"Params:\t max_lr={max_lr}\t LR_function={LR_function.__name__}")
    print('-------------------------------')
    CW_tanh_test(combined_model, mal_x_batch, mal_y_batch, removal_array, device='cuda', const=100.0, LR_function=LR_function,
                 max_lr=max_lr, min_lr=max_lr, update_step=500, first_phase_iters=20000,second_phase_iters = 5000 , confidence=0.1, eps=1e-3)

In [ ]:
import itertools

# Define hyperparameter grid
param_grid = {
    'max_lr': [0.01,0.02,0.04],
    'LR_function': [lr_cosine_half],
}


for params in itertools.product(*param_grid.values()):
    # Check if this set of parameters gives the best result
    print('**********************************************************************************************************************************************')
    max_lr, LR_function = params
    print(f"Params:\t max_lr={max_lr}\t LR_function={LR_function.__name__}")
    print('-------------------------------')
    CW_tanh_test(combined_model, mal_x_batch, mal_y_batch, removal_array, device='cuda', const=100.0, LR_function=LR_function,
                 max_lr=max_lr, min_lr=max_lr, update_step=500, first_phase_iters=20000,second_phase_iters = 5000 , confidence=0.1, eps=1e-3)

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim.lr_scheduler import LambdaLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.
    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def torch_arctanh(x, eps=1e-6):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))


# Define the learning rate schedules (same as earlier)
def lr_linear(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start + t * (lr_end - lr_start) / T

def lr_exponential(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start * (lr_end / lr_start) ** (t / T)

def lr_cosine(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_end + (lr_start - lr_end) * 0.5 * (1 + torch.cos(torch.tensor(np.pi * t / T)))

def lr_average(t, T, lr_start=1e-3, lr_end=0.9):
    return 0.5 * (lr_linear(t, T, lr_start, lr_end) + lr_exponential(t, T, lr_start, lr_end))

def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))



def CW(model, imgs, y, removal_array, device='cuda', initial_const = 1., binary_search_steps=8, cons_increase_factor=10., LR_function=lr_exponential,
                 max_lr=2., min_lr=1e-3,update_step=1000, first_phase_iters=20000,second_phase_iters = 10000 , confidence=0.1, eps=1e-4):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - max_lr: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform.
    - confidence: Confidence parameter for adversarial loss computation.
    """

    def grad_hook(grad):
        return grad * non_fixed_features_mask

    # Preparing tensors
    imgs_tensor = torch_arctanh(imgs, eps).clone().detach().to(dtype=torch.float32, device=device)
    imgs_tensor_original = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = torch.zeros_like(y.squeeze(), dtype=torch.int64, device=device)

    batch_size = imgs.shape[0]

    # Create masks for 0s and 1s
    zero_mask = (imgs_tensor == 0)  # Mask where values are 0
    one_mask = (imgs_tensor == 1)   # Mask where values are 1

    # Add eps to 0s and subtract eps from 1s
    imgs_tensor[zero_mask] += eps
    imgs_tensor[one_mask] -= eps

    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))


    # Initialize tracking variables
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestattack = imgs.clone().detach()

    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()
    o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

    model.eval()

    # Total number of iterations
    total_iters = first_phase_iters + second_phase_iters

    for outer_step in range(binary_search_steps):
        print('------------------------------------------------------------------------------------------------------------------------------------')
        print(CONST)

        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
        optimizer = torch.optim.Adam([modifier], lr=min_lr)
        modifier.register_hook(grad_hook)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestattack = imgs.clone().detach()
        best_modifier = torch.zeros_like(modifier).detach()  # Best modifier
        best_unsuccessfull_modifier = torch.zeros_like(modifier).detach()

        best_loss1 = torch.full((batch_size,), float('inf'), device=device)
        best_unsuccessfull_attack = imgs.clone().detach()
        success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for iteration in range(total_iters):

            # Phase 2: Fixed learning rate of 1e-3 (iterations 15,000 to 24,999)
            if iteration == first_phase_iters:
                print("************ second phase *************")
                output = model(o_bestattack)
                final_success = compare(output)
                final_output = torch.where(final_success.unsqueeze(-1), bestattack, best_unsuccessfull_attack)
                final_modifier = torch.where(final_success.unsqueeze(-1), best_modifier, best_unsuccessfull_modifier)

                modifier = torch.tensor(final_modifier.clone().detach(), requires_grad=True).to(device)
                optimizer = torch.optim.Adam([modifier], lr=1e-3)  # Use lr_min for 1000 iterations
                modifier.register_hook(grad_hook)

            if iteration < first_phase_iters:
                if iteration == 0 or (iteration + 1) % update_step == 0:
                    # Select your preferred learning rate schedule
                    current_lr = LR_function(iteration, first_phase_iters, min_lr, max_lr)  # You can switch to lr_linear, lr_exponential, or lr_cosine

                    # Update the optimizer's learning rate
                    for param_group in optimizer.param_groups:
                        param_group['lr'] = current_lr

            optimizer.zero_grad()

            # Attack Step: generate perturbed image
            modified_input = imgs_tensor + modifier
            newimg = torch.tanh(modified_input) * 0.5 + 0.5
            output = model(newimg)

            l2dist = torch.sum((newimg - imgs_tensor_original) ** 2, dim=1)

            real = output[:, 0]
            other = output[:, 1]

            loss1 = torch.clamp((other - real + confidence), min=0.0)
            loss = (CONST * loss1) + l2dist

            total_loss = torch.sum(loss)
            total_loss.backward()
            optimizer.step()

            with torch.no_grad():
                success_mask_inloop = compare(output)
                success_mask = torch.logical_or(success_mask_inloop, success_mask)
                o_success_mask  = torch.logical_or(o_success_mask, success_mask_inloop)

                update_mask = (l2dist < bestl2) & success_mask_inloop
                bestl2 = torch.where(update_mask, l2dist, bestl2)
                bestattack = torch.where(update_mask.unsqueeze(-1), newimg, bestattack)
                best_modifier = torch.where(update_mask.unsqueeze(-1), modifier.detach(), best_modifier)

                improved_mask = (loss1 < best_loss1)
                best_loss1 = torch.where(improved_mask, loss1, o_best_loss1)
                best_unsuccessfull_attack = torch.where(improved_mask.unsqueeze(-1), newimg, best_unsuccessfull_attack)
                best_unsuccessfull_modifier = torch.where(improved_mask.unsqueeze(-1), modifier.detach(), best_unsuccessfull_modifier)

                update_mask2 = (l2dist < o_bestl2) & success_mask_inloop
                o_bestl2 = torch.where(update_mask2, l2dist, o_bestl2)
                o_bestattack = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack)

                improved_mask2 = (loss1 < o_best_loss1)
                o_best_loss1 = torch.where(improved_mask2, loss1, o_best_loss1)
                o_best_unsuccessfull_attack = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack)


                if iteration == 0 or (iteration + 1) % 500 == 0:
                    current_lr = optimizer.param_groups[0]['lr']
                    current_momentum, current_momentum2 = optimizer.param_groups[0]['betas']
                    print(f"Iteration {iteration + 1:5}:\t Current LR = {current_lr:.4f} \t Current momentum = {current_momentum:.2f} \t Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t Loss = {total_loss.item():.4f} \t l2dist = {torch.mean(bestl2[success_mask]).item():.4f}")

                    print(f"Best results until now: \t success = {o_success_mask.sum().item()} \t l2dist = {torch.mean(o_bestl2[o_success_mask]).item():.4f}")

        failure_mask = ~success_mask
        upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])
        lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

        upper_limit_mask = upper_bound < 1e9
        CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
        CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor



    print('')
    output = model(o_bestattack)
    final_success = compare(output)
    print(f"FINAL RESULT :\t #success = {final_success.sum().item()} \t total Loss = {total_loss.item():.4f}\t success Loss = {loss[final_success].sum().item():.4f} \t success l2dist = {torch.mean(o_bestl2[final_success]).item():.4f}")
    print('')

    final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)
    results_rounded = evaluate_rounded_adversarial_examples(imgs, y, final_output, combined_model, removal_array)
    print('')

In [ ]:
CW(combined_model, mal_x_batch, mal_y_batch, removal_array, device='cuda', initial_const = 1., binary_search_steps=9, cons_increase_factor=10., LR_function=lr_cosine_half,
                 max_lr=0.02, min_lr=0.02,update_step=500, first_phase_iters=20000,second_phase_iters = 5000 , confidence=0.1, eps=1e-3)

## Step 7 — Watching the loss: modifier trajectories (from `analysis_Loss.ipynb`)

# dist : L2 norm

In [ ]:
sin_func_torch(torch.tensor(1 - 1e-2))

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import CyclicLR
from torch.optim.lr_scheduler import LambdaLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def sin_func_torch2(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (1 - torch.cos(input_tensor * torch.pi)) / 2.0

# Define warmup and decay functions
def lr_lambda(epoch):
    warmup_epochs = 100
    if epoch < warmup_epochs:
        return float(epoch) / float(max(1, warmup_epochs))
    return 1.0


def lr_range_test(model, imgs, y, removal_array, device='cuda', const=1.0, step_size_up=10000,step_size_down=10000,
                  lr_min=1e-6, lr_max=1e-1, num_iter=100, step_mode='linear', confidence=0.5):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    Parameters:
    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - lr_min: Minimum learning rate to start with.
    - lr_max: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform the LR test.
    - step_mode: 'linear' or 'exponential' mode for LR increase.
    - confidence: Confidence parameter for adversarial loss computation.

    Returns:
    - lrs: List of learning rates used during the test.
    - losses: List of corresponding losses.
    """
    model.eval()

    # Preparing tensors
    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    #imgs_tensor_org = imgs.to(device=device, dtype=torch.float32).clone().detach()

    imgs_tensor = imgs.to(device=device, dtype=torch.float32).clone().detach()

    # Small value eps
    eps = 1e-3

    # Create masks for 0s and 1s
    zero_mask = (imgs_tensor == 0)  # Mask where values are 0
    one_mask = (imgs_tensor == 1)   # Mask where values are 1

    # Add eps to 0s and subtract eps from 1s
    imgs_tensor[zero_mask] += eps
    imgs_tensor[one_mask] -= eps
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))



    modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
    optimizer = torch.optim.SGD([modifier], lr=lr_min,momentum=0.3)
    #optimizer = torch.optim.Adam([modifier], lr=lr_min,betas=(0.3,0.3))
    #optimizer = torch.optim.Adam([modifier], lr=lr_min, amsgrad=True,betas=(0.3,0.3))
    #optimizer = torch.optim.RMSprop([modifier], lr=lr_min, alpha=0.3)

    #scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

    def grad_hook(grad):
            return grad * non_fixed_features_mask
    modifier.register_hook(grad_hook)

    modifiers = []
    modified_inputs = []
    tensors = []
    imgs_tensors = []
    losses = []
    all_success = []
    batch_size = imgs.shape[0]
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_bestattack = imgs.clone().detach()

    for iteration in range(num_iter):

        optimizer.zero_grad()

        # Attack Step: generate perturbed image
        modified_input = imgs_tensor + modifier
        newimg = sin_func_torch(modified_input) # or newimg = sin_func_torch2(modified_input)
        output = model(newimg)

        #tensors.append(newimg.detach().clone())
        #imgs_tensors.append(imgs_tensor.detach().clone())
        #modifiers.append(modifier.detach().clone())
        #modified_inputs.append(modified_input.detach().clone())

        #l2dist = torch.sum((newimg - imgs_tensor_org) ** 2, dim=1)

        real = output[:, 0]
        other = output[:, 1]

        loss1 = (other - real + confidence)

        loss = (100 * loss1)

        total_loss = torch.sum(loss)

        total_loss.backward()

        # Get the gradient of `modifier`
        #modifier_grad = modifier.grad

        # Print the gradient at this step (or process it)
        #print(f"Step {iteration + 1}, Gradient: {modifier_grad.sum()}")
        #print(f"Step {iteration + 1}, Gradient: {torch.abs(modifier_grad).sum()}")


        optimizer.step()
        #scheduler.step()

        success_mask_inloop = compare(output)
        success_mask  = torch.logical_or(success_mask_inloop, success_mask)

        update_mask2 = success_mask_inloop
        o_bestattack = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack)


        # Record learning rate and loss
        current_lr = optimizer.param_groups[0]["lr"]
        if (iteration + 1) % 1== 0:

            losses.append(loss.detach().to('cpu'))
            all_success.append(success_mask.sum().item())

        # Debug: Print learning rate and loss
        if iteration == 0 or (iteration + 1) % 100== 0:
            current_momentum,current_momentum2 = 0.,0.
            print(f"Iteration {iteration + 1}: Current LR = {current_lr:.6f} \t Current success = {success_mask_inloop.sum().item()} Loss = {total_loss.item():.6f}")
            #print(f"Iteration {iteration + 1}: Current LR = {current_lr:.6f} \t Current momentum = {current_momentum:.6f} \t Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t Loss = {total_loss.item():.6f}")



    return modifiers,modified_inputs,tensors,imgs_tensors, losses,o_bestattack

In [ ]:
# optimizer = torch.optim.SGD([modifier], lr=lr_min,momentum=0.0)

for x_test, y_test in test_loader:
    x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
    mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
    break

In [ ]:
# loss1 = torch.clamp((other - real + confidence), min=0.0)
# oss = (100 * loss1) + l2dist

for x_test, y_test in test_loader:
    x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
    mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
    break

modifiers,modified_inputs,tensors,imgs_tensors, losses,adv = lr_range_test(combined_model, mal_x_batch[:1], mal_y_batch[:1], removal_array, device=device, const=100.0, step_size_up=20000,step_size_down=1,
                  lr_min=.0001, lr_max=.0001, num_iter=10000, step_mode='linear', confidence=0.1)

In [ ]:
# Convert list of tensors into a 2D NumPy array (iterations, samples)
# You may need to convert the tensors to numpy arrays first (if using PyTorch or TensorFlow)
loss_array = np.array([loss.numpy() for loss in losses])  # shape (40000, 100)

# Find the iteration where each sample has the minimum loss
min_loss_iteration = np.argmin(loss_array, axis=0)

# Now, min_loss_iteration contains the index of the iteration (row) where each sample has its minimum loss
print(min_loss_iteration)

In [ ]:
# Convert list of tensors into a 2D NumPy array (iterations, samples)
# You may need to convert the tensors to numpy arrays first (if using PyTorch or TensorFlow)
loss_array = np.array([loss.numpy() for loss in losses])  # shape (40000, 100)


# Plotting
plt.figure(figsize=(50, 10))
sample_idx = 0
plt.plot(loss_array[:, sample_idx], label=f'Sample {sample_idx + 1}')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Loss of Each Sample over Iterations')
plt.legend(loc='upper right', ncol=2, fontsize='small')
plt.show()

In [ ]:
# Convert the list of tensors into a tensor of size (1000, 10000)
stacked_tensor = torch.cat(modifiers, dim=0)

# Plot the changes for each tracked element
plt.figure(figsize=(20, 10))

tracked_indices = [0,1,2,3,4,5,6,7,8,9]
for idx in range(200):
    plt.plot(range(300), stacked_tensor[:,idx][5500:5800].to('cpu'), label=f"Element {idx}")

# Add titles and labels
plt.title("Changes in Tensor Elements over 1000 Iterations")
plt.xlabel("Iteration")
plt.ylabel("Element Value")
#plt.legend()

# Show the plot
plt.show()

In [ ]:
# Plotting
plt.figure(figsize=(50, 10))
for sample_idx in range(10):
    plt.plot(loss_array[:, sample_idx], label=f'Sample {sample_idx + 1}')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Loss of Each Sample over Iterations')
plt.legend(loc='upper right', ncol=2, fontsize='small')
plt.show()

# New Section

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import CyclicLR
from torch.optim.lr_scheduler import LambdaLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def sin_func_torch2(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (1 - torch.cos(input_tensor * torch.pi)) / 2.0

# Define warmup and decay functions
def lr_lambda(epoch):
    warmup_epochs = 100
    if epoch < warmup_epochs:
        return float(epoch) / float(max(1, warmup_epochs))
    return 1.0


def lr_range_test(model, imgs, y, removal_array, device='cuda', const=1.0, step_size_up=10000,step_size_down=10000,
                  lr_min=1e-6, lr_max=1e-1, num_iter=100, step_mode='linear', confidence=0.5):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    Parameters:
    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - lr_min: Minimum learning rate to start with.
    - lr_max: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform the LR test.
    - step_mode: 'linear' or 'exponential' mode for LR increase.
    - confidence: Confidence parameter for adversarial loss computation.

    Returns:
    - lrs: List of learning rates used during the test.
    - losses: List of corresponding losses.
    """
    model.eval()

    # Preparing tensors
    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    imgs_tensor_org = imgs.to(device=device, dtype=torch.float32).clone().detach()

    imgs_tensor = imgs.to(device=device, dtype=torch.float32).clone().detach()

    # Small value eps
    eps = 1e-4

    # Create masks for 0s and 1s
    zero_mask = (imgs_tensor == 0)  # Mask where values are 0
    one_mask = (imgs_tensor == 1)   # Mask where values are 1

    # Add eps to 0s and subtract eps from 1s
    imgs_tensor[zero_mask] += eps
    imgs_tensor[one_mask] -= eps
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))



    modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
    #optimizer = torch.optim.SGD([modifier], lr=lr_min,momentum=0.)
    optimizer = torch.optim.Adam([modifier], lr=lr_min)
    #optimizer = torch.optim.Adam([modifier], lr=lr_min, amsgrad=True,betas=(0.3,0.3))
    #optimizer = torch.optim.RMSprop([modifier], lr=lr_min, alpha=0.3)

    # Custom LambdaLR to increase learning rate every 1000 iterations
    def lr_lambda(iteration):
        # Every 1000 iterations, increase the learning rate by a factor of 1.1
        return 2 ** (iteration // 1000)


    scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

    def grad_hook(grad):
            return grad * non_fixed_features_mask
    modifier.register_hook(grad_hook)

    modifiers = []
    modified_inputs = []
    tensors = []
    imgs_tensors = []
    losses = []
    all_success = []
    batch_size = imgs.shape[0]
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_bestattack = imgs.clone().detach()

    for iteration in range(num_iter):

        optimizer.zero_grad()

        # Attack Step: generate perturbed image
        modified_input = imgs_tensor + modifier
        newimg = sin_func_torch(modified_input) # or newimg = sin_func_torch2(modified_input)
        output = model(newimg)




        l2dist = torch.sum((newimg - imgs_tensor_org) ** 2, dim=1)

        real = output[:, 0]
        other = output[:, 1]

        loss1 = (other - real + confidence)

        loss = (100 * loss1) + l2dist



        total_loss = torch.sum(loss)

        total_loss.backward()

        # Get the gradient of `modifier`
        #modifier_grad = modifier.grad

        # Print the gradient at this step (or process it)
        #print(f"Step {iteration + 1}, Gradient: {modifier_grad.sum()}")
        #print(f"Step {iteration + 1}, Gradient: {torch.abs(modifier_grad).sum()}")


        optimizer.step()
        scheduler.step()

        success_mask_inloop = compare(output)
        success_mask  = torch.logical_or(success_mask_inloop, success_mask)

        update_mask2 = success_mask_inloop
        o_bestattack = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack)


        '''with torch.no_grad():
          losses.append(loss.detach().to('cpu'))
          if iteration == 0 or (iteration + 1) % 100== 0:
              tensors.append(newimg.detach().clone())
              #imgs_tensors.append(imgs_tensor.detach().clone())
              #modifiers.append(modifier.detach().clone())
              #modified_inputs.append(modified_input.detach().clone())'''



        # Record learning rate and loss
        current_lr = optimizer.param_groups[0]["lr"]
        #if (iteration + 1) % 1== 0:
            #all_success.append(success_mask.sum().item())

        # Debug: Print learning rate and loss
        if iteration == 0 or (iteration + 1) % 100== 0:
            current_momentum,current_momentum2 = optimizer.param_groups[0]['betas']
            #print(f"Iteration {iteration + 1}: Current LR = {current_lr:.6f} \t Current success = {success_mask_inloop.sum().item()} Loss = {total_loss.item():.6f}")
            print(f"Iteration {iteration + 1}: Current LR = {current_lr:.6f} \t Current momentum = {current_momentum:.6f} \t Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t Loss = {total_loss.item():.6f}")



    return modifiers,modified_inputs,tensors,imgs_tensors, losses,o_bestattack

In [ ]:
# optimizer = torch.optim.SGD([modifier], lr=lr_min,0.3)
modifiers,modified_inputs,tensors,imgs_tensors, losses,adv = lr_range_test(combined_model, mal_x_batch, mal_y_batch, removal_array, device=device, const=100.0, step_size_up=1000,step_size_down=9000,
                  lr_min=.0001, lr_max=.001, num_iter=15000, step_mode='linear', confidence=0.1)

In [ ]:
loss_array = np.array([loss.numpy() for loss in losses])  # shape (40000, 100)


# Plotting
plt.figure(figsize=(50, 10))
sample_idx = 0
plt.plot(loss_array[:13000, sample_idx], label=f'Sample {sample_idx + 1}')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Loss of Each Sample over Iterations')
plt.legend(loc='upper right', ncol=2, fontsize='small')
plt.show()

In [ ]:
(tensors[112][0]-tensors[114][0])[:1000]

# New Section

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import CyclicLR

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0


def lr_range_test(model, imgs, y, removal_array, device='cuda', const=1.0, step_size_up=10000,step_size_down=10000,
                  lr_min=1e-6, lr_max=1e-1, num_iter=100, step_mode='linear', confidence=0.5):
    """
    Perform a Learning Rate Range Test for adversarial attack optimization using the Adam optimizer.

    Parameters:
    - model: The target model to attack.
    - imgs: The original images to be attacked.
    - y: The true labels of the images.
    - removal_array: A mask array specifying which features are fixed.
    - device: The device on which to perform computations.
    - lr_min: Minimum learning rate to start with.
    - lr_max: Maximum learning rate to end with.
    - num_iter: Number of iterations to perform the LR test.
    - step_mode: 'linear' or 'exponential' mode for LR increase.
    - confidence: Confidence parameter for adversarial loss computation.

    Returns:
    - lrs: List of learning rates used during the test.
    - losses: List of corresponding losses.
    """
    model.eval()

    # Preparing tensors
    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    imgs_tensor = imgs.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))



    modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)
    optimizer = torch.optim.Adam([modifier], lr=lr_min)

    scheduler = CyclicLR(optimizer, lr_min , lr_max, step_size_up=step_size_up,step_size_down=step_size_down)

    def grad_hook(grad):
            return grad * non_fixed_features_mask
    modifier.register_hook(grad_hook)

    lrs = []
    losses = []
    all_success = []
    batch_size = imgs.shape[0]
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_bestattack = imgs.clone().detach()

    for iteration in range(num_iter):
        optimizer.zero_grad()

        # Attack Step: generate perturbed image
        modified_input = imgs_tensor + modifier
        newimg = sin_func_torch(modified_input)
        output = model(newimg)

        l2dist = torch.sum((newimg - imgs_tensor) ** 2, dim=1)

        real = output[torch.arange(imgs_tensor.size(0)), labs_tensor]
        other = output[torch.arange(imgs_tensor.size(0)), labs_tensor + 1]

        loss1 = torch.clamp((other - real + confidence), min=0.0)

        loss = (100 * loss1) + l2dist

        total_loss = torch.sum(loss)

        total_loss.backward()
        optimizer.step()
        scheduler.step()

        success_mask_inloop = compare(output)
        success_mask  = torch.logical_or(success_mask_inloop, success_mask)

        update_mask2 = success_mask_inloop
        o_bestattack = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack)


        # Record learning rate and loss
        current_lr = optimizer.param_groups[0]["lr"]
        if (iteration + 1) % 1== 0:
            #lrs.append(current_lr)
            losses.append(loss.detach().to('cpu'))
            #all_success.append(success_mask.sum().item())

        # Debug: Print learning rate and loss
        if iteration == 0 or (iteration + 1) % 1000== 0:
            current_momentum,current_momentum2 = optimizer.param_groups[0]['betas']
            print(f"Iteration {iteration + 1}: Current LR = {current_lr:.6f} \t Current momentum = {current_momentum:.6f} \t Current success = {success_mask_inloop.sum().item()} \t success = {success_mask.sum().item()} \t Loss = {total_loss.item():.6f}")



    return losses

In [ ]:
for x_test, y_test in test_loader:
    x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
    mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
    break

losses = lr_range_test(combined_model, mal_x_batch[:100], mal_y_batch[:100], removal_array, device='cuda', const=100.0, step_size_up=10000,step_size_down=1,
                  lr_min=1e-6, lr_max=0.1, num_iter=10000, step_mode='linear', confidence=0.1)

## Step 8 — The penalty-function gallery (from `loss_functions.ipynb`)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the perturbation range
perturbations = np.linspace(0, 1, 200)

# Calculate L0 distance (number of non-zero elements)
# For the given problem, the L0 distance can be approximated
# by the number of perturbations where the perturbation is non-zero
# In this case, perturbations are continuous, so we can use a different approach
# Here we assume perturbations in the sense of binary presence

l0_distances = np.array([np.sum(p > 0) for p in perturbations])

# Calculate L2 norm
l2_norms = (perturbations ** 2)

# Plot L0 distance and L2 norm
plt.figure(figsize=(12, 6))

# Plot L0 distance
plt.subplot(1, 2, 1)
plt.plot(perturbations, l0_distances, label=r'$L_0$ Distance', color='blue')
plt.title(r'$L_0$ Norm')
plt.xlabel(r'Perturbation')
plt.ylabel(r'$L_0$ Distance')
plt.grid(True)
plt.legend()

# Plot L2 norm
plt.subplot(1, 2, 2)
plt.plot(perturbations, l2_norms, label=r'$L_2$ Distance', color='red')
plt.title(r'$L_2$ Norm')
plt.xlabel(r'Perturbation')
plt.ylabel(r'$L_2$ Distance')
plt.grid(True)
plt.legend()

plt.tight_layout()

plt.subplots_adjust(wspace=0.4)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the sine-based transformation function
def sin_func_np(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (np.sin((input_tensor - 0.5) * np.pi) + 1) / 2.0

# Define the derivative of the sine-based transformation function
def derivative_sin_func_np(input_tensor):
    """
    The derivative of the sine-based transformation function.
    """
    return (np.pi / 2.0) * np.cos((input_tensor - 0.5) * np.pi)

# Define parameters
active_modifier = np.linspace(-2, 2, 20001)
active_imgs_tensor = 0.
modified_input = active_imgs_tensor + active_modifier
newimg = sin_func_np(modified_input)
perturbation = np.where((active_imgs_tensor==1.), (active_imgs_tensor - newimg), (newimg - active_imgs_tensor))

# Calculate sparsity loss
sparsity_loss = perturbation

# Calculate the derivative values
derivative_values = derivative_sin_func_np(modified_input)

# Plot sparsity loss and its derivative
plt.figure(figsize=(14, 6))

# Plot the sparsity loss
plt.subplot(1, 2, 1)
plt.plot(active_modifier, sparsity_loss, label=r'$\frac{1}{2} (\sin((w_i - 0.5) * \pi) + 1)$', color='blue')
plt.title(r'$\frac{1}{2} (\sin((w_i - 0.5) * pi + 1)$ Transformation')
plt.xlabel(r'$w_i$')
plt.ylabel(r'$\frac{1}{2} (\sin((w_i - 0.5) * pi + 1)$')
plt.grid(True)
plt.legend()

# Plot the derivative of the function
plt.subplot(1, 2, 2)
plt.plot(active_modifier, derivative_values, label=r'Derivative of $\frac{1}{2} (\sin((w_i - 0.5) * \pi) + 1)$', color='red')
plt.title(r'Derivative of $\frac{1}{2} (\sin((w_i - 0.5) * \pi) + 1)$')
plt.xlabel(r'$w_i$')
plt.ylabel(r'Derivative')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def sin_func_np(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (np.sin((input_tensor - 0.5) * np.pi) + 1) / 2.0

# Define parameters
active_modifier = np.linspace(-2, 2, 20001)
active_imgs_tensor = 0.
modified_input = active_imgs_tensor + active_modifier
newimg = sin_func_np(modified_input)
perturbation = np.where((active_imgs_tensor==1.), (active_imgs_tensor - newimg), (newimg - active_imgs_tensor) )

# Calculate sparsity loss
sparsity_loss = ((perturbation))
#sparsity_loss = ((perturbation)**2)


# Plot sparsity loss
plt.figure(figsize=(12, 6))
plt.plot(active_modifier, sparsity_loss, label=r'$\frac{1}{2} (\sin((w_i - 0.5) * pi + 1)$')
plt.title(r'$\frac{1}{2} (\sin((w_i - 0.5) * pi + 1)$ Transformation')
plt.xlabel(r'$w_i$')
plt.ylabel(r'$\frac{1}{2} (\sin((w_i - 0.5) * pi + 1)$')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the function to be plotted
def transformation(w):
    return 0.5 * (np.tanh(w) + 1)



# Generate values for w from -10 to 10
w_values = np.linspace(-10, 10, 1000)
transformed_values = transformation(w_values)


# Plot the function
plt.figure(figsize=(12, 6))
plt.plot(w_values, transformed_values, label=r'$\frac{1}{2} (\tanh(w_i) + 1)$', color='blue')
plt.axhline(0, color='gray', linestyle='--', linewidth=0.5)
plt.axvline(0, color='gray', linestyle='--', linewidth=0.5)
plt.xlabel(r'$w_i$')
plt.ylabel(r'$\frac{1}{2} (\tanh(w_i) + 1)$')
plt.title(r'$\frac{1}{2} (\tanh(w_i) + 1)$ Transformation')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the function to be plotted
def transformation(w):
    return 0.5 * (np.tanh(w) + 1)

# Define the derivative of the function
def derivative_transformation(w):
    return 0.5 * (1 - np.tanh(w)**2)

# Generate values for w from -10 to 10
w_values = np.linspace(-10, 10, 1000)
transformed_values = transformation(w_values)
derivative_values = derivative_transformation(w_values)

# Plot the function and its derivative
plt.figure(figsize=(12, 6))

# Plot the original function
plt.subplot(1, 2, 1)
plt.plot(w_values, transformed_values, label=r'$\frac{1}{2} (\tanh(w_i) + 1)$', color='blue')
plt.axhline(0, color='gray', linestyle='--', linewidth=0.5)
plt.axvline(0, color='gray', linestyle='--', linewidth=0.5)
plt.xlabel(r'$w_i$')
plt.ylabel(r'$\frac{1}{2} (\tanh(w_i) + 1)$')
plt.title(r'$\frac{1}{2} (\tanh(w_i) + 1)$ Transformation')
plt.grid(True)
plt.legend()

# Plot the derivative
plt.subplot(1, 2, 2)
plt.plot(w_values, derivative_values, label=r'Derivative of $\frac{1}{2} (\tanh(w_i) + 1)$', color='red')
plt.axhline(0, color='gray', linestyle='--', linewidth=0.5)
plt.axvline(0, color='gray', linestyle='--', linewidth=0.5)
plt.xlabel(r'$w_i$')
plt.ylabel(r'Derivative')
plt.title(r'Derivative of $\frac{1}{2} (\tanh(w_i) + 1)$')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def sin_func_np(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (np.sin((input_tensor - 0.5) * np.pi) + 1) / 2.0

# Define parameters
active_modifier = np.linspace(-2, 2, 20001)
modified_input = 0. + active_modifier
newimg = sin_func_np(modified_input)
perturbation = (newimg - 0)


# Calculate sparsity loss
sparsity_loss = np.tanh(perturbation)
sparsity_loss2 = np.tanh(2 * perturbation)
sparsity_loss3 = np.tanh(4 * perturbation)
#sparsity_loss = ((perturbation)**2)


# Plot sparsity loss
plt.figure(figsize=(12, 6))
plt.plot(active_modifier, sparsity_loss, label='Sparsity-Inducing Loss1')
plt.plot(active_modifier, sparsity_loss2, label='Sparsity-Inducing Loss2')
plt.plot(active_modifier, sparsity_loss3, label='Sparsity-Inducing Loss4')
plt.title('Sparsity-Inducing Loss vs. Perturbation')
plt.xlabel('Perturbation')
plt.ylabel('Sparsity-Inducing Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Convert perturbation to a PyTorch tensor
active_modifier = torch.tensor(active_modifier, requires_grad=True)

modified_input = 0. + active_modifier
newimg = sin_func_torch(modified_input)
perturbation_tensor = (newimg - 0)

# Calculate sparsity loss in PyTorch
sparsity_loss_tensor = torch.tanh(perturbation_tensor)

# Calculate the gradient (numerical derivative) of the sparsity loss
sparsity_loss_tensor.backward(torch.ones_like(perturbation_tensor))
gradient = active_modifier.grad

# Convert perturbation to a PyTorch tensor
active_modifier2 = torch.tensor(active_modifier, requires_grad=True)

modified_input2 = 0. + active_modifier2
newimg2 = sin_func_torch(modified_input2)
perturbation_tensor2 = (newimg2 - 0)

# Calculate sparsity loss in PyTorch
sparsity_loss_tensor2 = torch.tanh(2 * perturbation_tensor2)

# Calculate the gradient (numerical derivative) of the sparsity loss
sparsity_loss_tensor2.backward(torch.ones_like(perturbation_tensor2))
gradient2 = active_modifier2.grad

# Convert perturbation to a PyTorch tensor
active_modifier3 = torch.tensor(active_modifier, requires_grad=True)

modified_input3 = 0. + active_modifier3
newimg3 = sin_func_torch(modified_input3)
perturbation_tensor3 = (newimg3 - 0)

# Calculate sparsity loss in PyTorch
sparsity_loss_tensor3 = torch.tanh(4 * perturbation_tensor3)

# Calculate the gradient (numerical derivative) of the sparsity loss
sparsity_loss_tensor3.backward(torch.ones_like(perturbation_tensor3))
gradient3 = active_modifier3.grad

# Plot the gradient
plt.figure(figsize=(12, 6))
plt.plot(active_modifier.detach().numpy(), gradient.detach().numpy(), label='Numerical Derivative (Gradient)')
plt.plot(active_modifier.detach().numpy(), gradient2.detach().numpy(), label='Numerical Derivative (Gradient2)')
plt.plot(active_modifier.detach().numpy(), gradient3.detach().numpy(), label='Numerical Derivative (Gradient3)')
plt.title('Numerical Derivative of Sparsity-Inducing Loss')
plt.xlabel('Perturbation')
plt.ylabel('Gradient')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def sin_func_np(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (np.sin((input_tensor - 0.5) * np.pi) + 1) / 2.0

# Define parameters
active_modifier = np.linspace(-2, 2, 20001)
modified_input = 0. + active_modifier
newimg = sin_func_np(modified_input)
perturbation = (newimg - 0)


# Calculate sparsity loss
eps = 1e-2
sparsity_loss = (1 - (perturbation - 1)**2+eps)**0.5 - eps**0.5
#sparsity_loss = ((perturbation)**2)


# Plot sparsity loss
plt.figure(figsize=(12, 6))
plt.plot(active_modifier, sparsity_loss, label='Sparsity-Inducing Loss')
plt.title('Sparsity-Inducing Loss vs. Perturbation')
plt.xlabel('Perturbation')
plt.ylabel('Sparsity-Inducing Loss')
plt.legend()
plt.grid(True)
plt.show()

# Convert perturbation to a PyTorch tensor
active_modifier = torch.tensor(active_modifier, requires_grad=True)

modified_input = 0. + active_modifier
newimg = sin_func_torch(modified_input)
perturbation_tensor = (newimg - 0)

# Calculate sparsity loss in PyTorch
sparsity_loss_tensor = (1 - (perturbation_tensor - 1)**2 + eps)**0.5 - eps**0.5

# Calculate the gradient (numerical derivative) of the sparsity loss
sparsity_loss_tensor.backward(torch.ones_like(perturbation_tensor))
gradient = active_modifier.grad

# Plot the gradient
plt.figure(figsize=(12, 6))
plt.plot(active_modifier.detach().numpy(), gradient.detach().numpy(), label='Numerical Derivative (Gradient)')
plt.title('Numerical Derivative of Sparsity-Inducing Loss')
plt.xlabel('Perturbation')
plt.ylabel('Gradient')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
print(f"Perturbation at index 20000 (expected 0): {perturbation[10000]}")
print(f"Gradient at index 20000 (corresponding to perturbation=0): {gradient[10000]}")
max_grad_index = gradient.argmax()
max_grad_value = gradient[max_grad_index]
corresponding_perturbation = perturbation[max_grad_index]
print(f"Perturbation corresponding to the maximum gradient: {corresponding_perturbation}")
print(f"Maximum gradient: {max_grad_value}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def sin_func_np(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (np.sin((input_tensor - 0.5) * np.pi) + 1) / 2.0

# Define parameters
active_modifier = np.linspace(-2, 2, 20001)
modified_input = 0. + active_modifier
newimg = sin_func_np(modified_input)
perturbation = (newimg - 0)

# Calculate sparsity loss
sparsity_loss = ((perturbation)**2)
#sparsity_loss = ((perturbation)**2)


# Plot sparsity loss
plt.figure(figsize=(12, 6))
plt.plot(active_modifier, sparsity_loss, label='Sparsity-Inducing Loss')
plt.title('Sparsity-Inducing Loss vs. Perturbation')
plt.xlabel('Perturbation')
plt.ylabel('Sparsity-Inducing Loss')
plt.legend()
plt.grid(True)
plt.show()

# Convert perturbation to a PyTorch tensor
active_modifier = torch.tensor(active_modifier, requires_grad=True)

modified_input = 0. + active_modifier
newimg = sin_func_torch(modified_input)
perturbation_tensor = (newimg - 0)

# Calculate sparsity loss in PyTorch
sparsity_loss_tensor = ((perturbation_tensor)**2)

# Calculate the gradient (numerical derivative) of the sparsity loss
sparsity_loss_tensor.backward(torch.ones_like(perturbation_tensor))
gradient = active_modifier.grad

# Plot the gradient
plt.figure(figsize=(12, 6))
plt.plot(active_modifier.detach().numpy(), gradient.detach().numpy(), label='Numerical Derivative (Gradient)')
plt.title('Numerical Derivative of Sparsity-Inducing Loss')
plt.xlabel('Perturbation')
plt.ylabel('Gradient')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def sin_func_np(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (np.sin((input_tensor - 0.5) * np.pi) + 1) / 2.0

# Define parameters
active_modifier = np.linspace(-2, 2, 20001)
active_imgs_tensor = 0.
modified_input = active_imgs_tensor + active_modifier
newimg = sin_func_np(modified_input)
perturbation = np.where((active_imgs_tensor==1.), (active_imgs_tensor - newimg), (newimg - active_imgs_tensor) )

# Calculate sparsity loss
sparsity_loss = ((perturbation))
#sparsity_loss = ((perturbation)**2)


# Plot sparsity loss
plt.figure(figsize=(12, 6))
plt.plot(active_modifier, sparsity_loss, label='Sparsity-Inducing Loss')
plt.title('Sparsity-Inducing Loss vs. Perturbation')
plt.xlabel('Perturbation')
plt.ylabel('Sparsity-Inducing Loss')
plt.legend()
plt.grid(True)
plt.show()

# Convert perturbation to a PyTorch tensor
active_modifier = torch.tensor(active_modifier, requires_grad=True)

modified_input = 1. + active_modifier
newimg = sin_func_torch(modified_input)
perturbation_tensor = (newimg - 1)

# Calculate sparsity loss in PyTorch
sparsity_loss_tensor = (torch.abs(perturbation_tensor))

# Calculate the gradient (numerical derivative) of the sparsity loss
sparsity_loss_tensor.backward(torch.ones_like(perturbation_tensor))
gradient = active_modifier.grad

# Plot the gradient
plt.figure(figsize=(12, 6))
plt.plot(active_modifier.detach().numpy(), gradient.detach().numpy(), label='Numerical Derivative (Gradient)')
plt.title('Numerical Derivative of Sparsity-Inducing Loss')
plt.xlabel('Perturbation')
plt.ylabel('Gradient')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

def sin_func_torch(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (torch.sin((input_tensor - 0.5) * torch.pi) + 1) / 2.0

def sin_func_np(input_tensor):
    """
    A sine-based transformation applied to the input tensor.

    The function applies a sine transformation and rescales the output to [0, 1].
    """
    return (np.sin((input_tensor - 0.5) * np.pi) + 1) / 2.0

# Define parameters
active_modifier = np.linspace(-2, 2, 20001)
modified_input = 1. + active_modifier
newimg = sin_func_np(modified_input)
perturbation = (newimg - 1)

# Calculate sparsity loss
sparsity_loss = ((perturbation)*np.sign(perturbation))
#sparsity_loss = ((perturbation)**2)


# Plot sparsity loss
plt.figure(figsize=(12, 6))
plt.plot(active_modifier, sparsity_loss, label='Sparsity-Inducing Loss')
plt.title('Sparsity-Inducing Loss vs. Perturbation')
plt.xlabel('Perturbation')
plt.ylabel('Sparsity-Inducing Loss')
plt.legend()
plt.grid(True)
plt.show()

# Convert perturbation to a PyTorch tensor
active_modifier = torch.tensor(active_modifier, requires_grad=True)

modified_input = 1. + active_modifier
newimg = sin_func_torch(modified_input)
perturbation_tensor = (newimg - 1)

# Calculate sparsity loss in PyTorch
sparsity_loss_tensor = ((perturbation_tensor)*torch.sign(perturbation_tensor))

# Calculate the gradient (numerical derivative) of the sparsity loss
sparsity_loss_tensor.backward(torch.ones_like(perturbation_tensor))
gradient = active_modifier.grad

# Plot the gradient
plt.figure(figsize=(12, 6))
plt.plot(active_modifier.detach().numpy(), gradient.detach().numpy(), label='Numerical Derivative (Gradient)')
plt.title('Numerical Derivative of Sparsity-Inducing Loss')
plt.xlabel('Perturbation')
plt.ylabel('Gradient')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

# Define parameters
epsilon = 0.01
perturbation = np.linspace(-1, 1, 20001)
perturbation = (np.sin((perturbation - 0.5) * np.pi) + 1) / 2.0

# Calculate sparsity loss
sparsity_loss = np.tanh(10 * (perturbation)*np.sign(perturbation))

# Plot sparsity loss
plt.figure(figsize=(12, 6))
plt.plot(perturbation, sparsity_loss, label='Sparsity-Inducing Loss')
plt.title('Sparsity-Inducing Loss vs. Perturbation')
plt.xlabel('Perturbation')
plt.ylabel('Sparsity-Inducing Loss')
plt.legend()
plt.grid(True)
plt.show()

# Convert perturbation to a PyTorch tensor
perturbation_tensor = torch.tensor(perturbation, requires_grad=True)

# Calculate sparsity loss in PyTorch
sparsity_loss_tensor = torch.tanh(10 * (perturbation_tensor)*torch.sign(perturbation_tensor))

# Calculate the gradient (numerical derivative) of the sparsity loss
sparsity_loss_tensor.backward(torch.ones_like(perturbation_tensor))
gradient = perturbation_tensor.grad

# Plot the gradient
plt.figure(figsize=(12, 6))
plt.plot(perturbation, gradient.detach().numpy(), label='Numerical Derivative (Gradient)')
plt.title('Numerical Derivative of Sparsity-Inducing Loss')
plt.xlabel('Perturbation')
plt.ylabel('Gradient')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

# Define parameters
epsilon = 0.01
perturbation = np.linspace(-1, 1, 10001)

# Calculate sparsity loss
sparsity_loss = perturbation / (perturbation + epsilon*np.sign(perturbation))

# Plot sparsity loss
plt.figure(figsize=(12, 6))
plt.plot(perturbation, sparsity_loss, label='Sparsity-Inducing Loss')
plt.title('Sparsity-Inducing Loss vs. Perturbation')
plt.xlabel('Perturbation')
plt.ylabel('Sparsity-Inducing Loss')
plt.legend()
plt.grid(True)
plt.show()

# Convert perturbation to a PyTorch tensor
perturbation_tensor = torch.tensor(perturbation, requires_grad=True)

# Calculate sparsity loss in PyTorch
sparsity_loss_tensor = perturbation_tensor/ (perturbation_tensor + epsilon * torch.sign(perturbation_tensor))

# Calculate the gradient (numerical derivative) of the sparsity loss
sparsity_loss_tensor.backward(torch.ones_like(perturbation_tensor))
gradient = perturbation_tensor.grad

# Plot the gradient
plt.figure(figsize=(12, 6))
plt.plot(perturbation, gradient.detach().numpy(), label='Numerical Derivative (Gradient)')
plt.title('Numerical Derivative of Sparsity-Inducing Loss')
plt.xlabel('Perturbation')
plt.ylabel('Gradient')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

# Define parameters
epsilon = 0.01
perturbation = np.linspace(-1, 1, 10001)

# Calculate sparsity loss
sparsity_loss = perturbation**2 / (perturbation**2 + epsilon)

# Plot sparsity loss
plt.figure(figsize=(12, 6))
plt.plot(perturbation, sparsity_loss, label='Sparsity-Inducing Loss')
plt.title('Sparsity-Inducing Loss vs. Perturbation')
plt.xlabel('Perturbation')
plt.ylabel('Sparsity-Inducing Loss')
plt.legend()
plt.grid(True)
plt.show()

# Convert perturbation to a PyTorch tensor
perturbation_tensor = torch.tensor(perturbation, requires_grad=True)

# Calculate sparsity loss in PyTorch
sparsity_loss_tensor = perturbation_tensor**2 / (perturbation_tensor**2 + epsilon)

# Calculate the gradient (numerical derivative) of the sparsity loss
sparsity_loss_tensor.backward(torch.ones_like(perturbation_tensor))
gradient = perturbation_tensor.grad

# Plot the gradient
plt.figure(figsize=(12, 6))
plt.plot(perturbation, gradient.detach().numpy(), label='Numerical Derivative (Gradient)')
plt.title('Numerical Derivative of Sparsity-Inducing Loss')
plt.xlabel('Perturbation')
plt.ylabel('Gradient')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the function with epsilon for smoothing
epsilon = 1e-1
perturbation = np.linspace(-1, 1, 500)

# Define the custom function
custom_function = np.sqrt(perturbation**2 / (perturbation**2 + epsilon**2))

# Convert perturbation to a PyTorch tensor
perturbation_tensor = torch.tensor(perturbation, requires_grad=True)

# Calculate sparsity loss in PyTorch
sparsity_loss_tensor = torch.abs(perturbation_tensor) / (torch.abs(perturbation_tensor) + epsilon)

# Calculate the gradient (numerical derivative) of the sparsity loss
sparsity_loss_tensor.backward(torch.ones_like(perturbation_tensor))
gradient = perturbation_tensor.grad

# Plot the custom function
plt.figure(figsize=(12, 6))
plt.plot(perturbation, custom_function, label='Custom Function')
plt.title('Custom Function vs. Perturbation')
plt.xlabel('Perturbation')
plt.ylabel('Function Value')
plt.legend()
plt.grid(True)
plt.show()

# Plot the gradient of the custom function
plt.figure(figsize=(12, 6))
plt.plot(perturbation, gradient_custom_function, label='Gradient of Custom Function', color='orange')
plt.title('Gradient of Custom Function vs. Perturbation')
plt.xlabel('Perturbation')
plt.ylabel('Gradient')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Define a smoothing parameter for softplus
beta = 10

# Softplus approximation of the absolute value
softplus_smooth = (1/beta) * np.log(1 + np.exp(beta * perturbation)) + (1/beta) * np.log(1 + np.exp(-beta * perturbation))

# Calculate smooth sparsity loss
smooth_sparsity_loss = softplus_smooth / (softplus_smooth + epsilon)

# Plot the original and smooth sparsity loss functions
plt.figure(figsize=(12, 6))
plt.plot(perturbation, sparsity_loss, label='Original Sparsity-Inducing Loss', linestyle='--')
plt.plot(perturbation, smooth_sparsity_loss, label='Smooth Sparsity-Inducing Loss (Softplus Approximation)')
plt.title('Original vs. Smooth Sparsity-Inducing Loss')
plt.xlabel('Perturbation')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# Convert perturbation to a PyTorch tensor for smooth loss
perturbation_tensor = torch.tensor(perturbation, requires_grad=True)

# Calculate smooth sparsity loss in PyTorch
softplus_smooth_tensor = (1/beta) * torch.log(1 + torch.exp(beta * perturbation_tensor)) + (1/beta) * torch.log(1 + torch.exp(-beta * perturbation_tensor))
smooth_sparsity_loss_tensor = softplus_smooth_tensor / (softplus_smooth_tensor + epsilon)

# Calculate the gradient (numerical derivative) of the smooth sparsity loss
smooth_sparsity_loss_tensor.backward(torch.ones_like(perturbation_tensor))
smooth_gradient = perturbation_tensor.grad

# Plot the gradient for the smooth function
plt.figure(figsize=(12, 6))
plt.plot(perturbation, gradient.detach().numpy(), label='Original Gradient', linestyle='--')
plt.plot(perturbation, smooth_gradient.detach().numpy(), label='Smooth Gradient (Softplus Approximation)')
plt.title('Original vs. Smooth Gradient of Sparsity-Inducing Loss')
plt.xlabel('Perturbation')
plt.ylabel('Gradient')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the perturbation range
perturbation = np.linspace(-1, 1, 500)

# Define the loss functions
alpha = 10
epsilon = 1e-2

# Log-Sum-Exp (LSE) approximation
lse_loss = np.log(1 + alpha * perturbation)

# Sparsity-inducing approximation
sparsity_loss = np.abs(perturbation) / (np.abs(perturbation) + epsilon)

# Softplus approximation
softplus_loss = np.log(1 + perturbation / epsilon)

# Plot the losses
plt.figure(figsize=(10, 6))
#plt.plot(perturbation, lse_loss, label='Log-Sum-Exp (LSE) Approximation', linestyle='-', color='blue')
plt.plot(perturbation, sparsity_loss, label='Sparsity-Inducing Approximation', linestyle='--', color='red')
#plt.plot(perturbation, softplus_loss, label='Softplus Approximation', linestyle='-.', color='green')

plt.title('Comparison of Loss Functions for Perturbation [0, 1]')
plt.xlabel('Perturbation')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Calculate the gradients (derivatives) of the loss functions with respect to perturbation

epsilon = 1e-2

# Gradient of Log-Sum-Exp (LSE) approximation
lse_grad = 1 / (1 + alpha * perturbation)

# Gradient of Sparsity-inducing approximation
sparsity_grad = epsilon / (np.abs(perturbation) + epsilon)**2

# Gradient of Softplus approximation
softplus_grad = 1 / (perturbation + epsilon)

# Plot the gradients
plt.figure(figsize=(10, 6))
#plt.plot(perturbation, lse_grad, label='Gradient of LSE Approximation', linestyle='-', color='blue')
plt.plot(perturbation, sparsity_grad, label='Gradient of Sparsity-Inducing Approximation', linestyle='--', color='red')
#plt.plot(perturbation, softplus_grad, label='Gradient of Softplus Approximation', linestyle='-.', color='green')

plt.title('Gradients of Loss Functions for Perturbation [0, 1]')
plt.xlabel('Perturbation')
plt.ylabel('Gradient')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the function and its derivative
def sparsity_loss(perturbation, epsilon=0.1):
    return perturbation / (perturbation + epsilon)

def sparsity_loss_derivative(perturbation, epsilon=0.1):
    return epsilon / (perturbation + epsilon)**2

# Define the range for perturbation
perturbation = np.linspace(-1, 1, 400)
epsilon = 0.1

# Compute the function and its derivative
loss = sparsity_loss(perturbation, epsilon)
loss_derivative = sparsity_loss_derivative(perturbation, epsilon)

# Create the plots
plt.figure(figsize=(12, 6))

# Plot the function
plt.subplot(1, 2, 1)
plt.plot(perturbation, loss, label='Sparsity Loss')
plt.title('Sparsity Loss Function')
plt.xlabel('Perturbation')
plt.ylabel('Sparsity Loss')
plt.grid(True)
plt.legend()

# Plot the derivative
plt.subplot(1, 2, 2)
plt.plot(perturbation, loss_derivative, label='Derivative of Sparsity Loss', color='r')
plt.title('Derivative of Sparsity Loss')
plt.xlabel('Perturbation')
plt.ylabel('Derivative')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()